# **_De novo_ glycoprotein binder design with RFDiffusion3**

## Introduction

Since RFDiffusion3 (RFD3) can diffuse both backbone and side-chain atoms simultaneously while conditioning directly on target biomolecules and non-protein components, it can also support _de novo_ protein design around glycoproteins by explicitly modeling glycans and post-translational modifications as all-atom non-protein components during diffusion. Unlike earlier backbone-only versions of RFDiffusion that ignored carbohydrate chains, often resulting in designed binders that sterically clashed with native glycosylation shields, RFD3 models side chains and non-protein atoms from the start of the denoising trajectory

<figure>
<center>
<img src='https://raw.githubusercontent.com/pb3lab/ibm3410/main/img/rfd3_img04.png'/>
<figcaption><b>Fig. 1</b>. Comparison of the folding process of binders and the diffusion process for generating binders against a given target protein.
<br><i>Image Credits: RosettaCommons Foundry</i></figcaption></center>
</figure>

In this tutorial, we will explore how to perform _de novo_ binder design with RFD3 against glycosylated CD28. This will be followed by improving their sequences with **SolubleMPNN** for protein-only targets, or **LigandMPNN** when an interface ligand is present, and _in silico_ validate that the binders fold and bind to the target hotspot via one of many protein structure prediction architectures (including **AlphaFold3**, **OpenFold3**, **Boltz-2** and **RoseTTAFold3**) as an oracle.

In [ ]:
%%time
import os, sys, re, time, glob, json, shlex, shutil, pathlib, tempfile, warnings, subprocess
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
warnings.filterwarnings('ignore')

try:
  from rich.console import Console
  from rich.progress import Progress, TextColumn, BarColumn, DownloadColumn, TransferSpeedColumn, TimeRemainingColumn
except ModuleNotFoundError:
  !pip -q install rich
  from rich.console import Console
  from rich.progress import Progress, TextColumn, BarColumn, DownloadColumn, TransferSpeedColumn, TimeRemainingColumn

#@title Setup environment
#@markdown Run this only **once**. Typically takes ~3 min.
#@markdown - **Foundry**: RFD3 + built-in MPNN (SolubleMPNN / LigandMPNN / ProteinMPNN) + optional RF3 + HBPLUS
#@markdown - **Folding model**: Boltz2, RosettaFold3, AlphaFold3, or OpenFold3

folding_model = "AlphaFold3" #@param ["RosettaFold3", "Boltz2", "AlphaFold3", "OpenFold3"] {type:"string"}

#@markdown Save Foundry checkpoints (RFD3, MPNN, RF3) to Google Drive to skip re-downloading across sessions
persist_foundry_checkpoints_to_drive = False #@param {type:"boolean"}
#@markdown Save AF3/OpenFold3 compiled XLA cache to Google Drive to skip ~53 s recompile across sessions
persist_af3_cache_to_drive = False #@param {type:"boolean"}
#@markdown Additional options
force_redownload = False #@param {type:"boolean"}
force_reinstall = False #@param {type:"boolean"}
check_GPU_runtime = True #@param {type:"boolean"}

console = Console()

if force_reinstall:
  os.environ['UV_VENV_CLEAR'] = '1'

# Silence noisy subprocess warnings & unused local-mirror warnings from atomworks
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['CCD_MIRROR_PATH'] = '/tmp'
os.environ['PDB_MIRROR_PATH'] = '/tmp'

if check_GPU_runtime:
  try:
    res = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=True)
    print(res.stdout.strip(), flush=True)
  except Exception as e:
    print(f"No GPU available ({e}). Will not proceed with setup.", file=sys.stderr, flush=True)
    raise SystemExit(1)

# Model flags
USE_BOLTZ = (folding_model == "Boltz2")
USE_RF3   = (folding_model == "RosettaFold3")
USE_AF3   = (folding_model == "AlphaFold3")
USE_OF3   = (folding_model == "OpenFold3")
USE_AF3_STACK = USE_AF3 or USE_OF3

# Mount Google Drive once on the main thread if any persistence option is enabled
drive_mounted = False
if persist_foundry_checkpoints_to_drive or (USE_AF3_STACK and persist_af3_cache_to_drive):
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_mounted = True
  except Exception as e:
    console.print(f"[yellow][Drive] Mount failed, using local /content storage: {e}[/yellow]")

ENVS_DIR = '/content/envs'
FOUNDRY_CKPTS = (
    '/content/drive/MyDrive/foundry_checkpoints'
    if (persist_foundry_checkpoints_to_drive and drive_mounted)
    else '/content/foundry/checkpoints'
)
HBPLUS_BIN = '/usr/local/bin/hbplus'
AF3_DIR = '/content/af3'
AF3_VERSION = '3.1.18'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
AF3_CACHE_DIR = (
    '/content/drive/MyDrive/.af3_cache'
    if (USE_AF3_STACK and persist_af3_cache_to_drive and drive_mounted)
    else os.path.join(tempfile.gettempdir(), 'af3_cache')
)

os.environ.update({
    'FOLDING_MODEL': folding_model,
    'USE_BOLTZ': str(USE_BOLTZ),
    'USE_RF3': str(USE_RF3),
    'USE_AF3': str(USE_AF3),
    'USE_OF3': str(USE_OF3),
    'ENVS_DIR': ENVS_DIR,
    'FOUNDRY_CHECKPOINT_DIRS': FOUNDRY_CKPTS,
    'HBPLUS_PATH': HBPLUS_BIN,
})
if USE_AF3_STACK:
  os.makedirs(AF3_CACHE_DIR, exist_ok=True)
  os.environ['AF3_CACHE_DIR'] = AF3_CACHE_DIR

# Bootstrap uv, aria2c, and gcc (for HBPLUS)
if shutil.which('uv') is None:
  subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
apt_pkgs = [pkg for cmd, pkg in [('aria2c', 'aria2'), ('gcc', 'build-essential')] if shutil.which(cmd) is None]
if apt_pkgs:
  subprocess.run(['apt-get', '-qq', 'update'], check=True)
  subprocess.run(['apt-get', '-qq', 'install', '-y', *apt_pkgs], check=True)

os.makedirs(ENVS_DIR, exist_ok=True)
os.chdir('/content')


def run_command_with_env(env: str, cmd: list[str], quiet=False, check=True):
  """Run a command inside a uv virtual environment and stream live output to Colab."""
  env_path = pathlib.Path(ENVS_DIR) / env
  if not env_path.exists():
    raise FileNotFoundError(f"uv environment '{env}' does not exist at {env_path}")
  if not quiet:
    print(f"[{env}] {' '.join(cmd)}", flush=True)
  sub_env = os.environ.copy()
  sub_env['VIRTUAL_ENV'] = str(env_path)
  sub_env['PATH'] = f"{env_path}/bin:{sub_env['PATH']}"
  sub_env['PYTHONUNBUFFERED'] = '1'

  proc = subprocess.Popen(
      cmd, env=sub_env,
      stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
      text=True, bufsize=1,
  )
  for line in proc.stdout:
    if not quiet:
      print(line, end='', flush=True)
  rc = proc.wait()
  if check and rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)
  return proc


def _download_with_rich(url: str, dest: pathlib.Path, progress: Progress, use_aria2: bool = False, min_bytes: int = 1000):
  """Download a file using a shared rich.progress bar, preserving final speed and elapsed time."""
  dest.parent.mkdir(parents=True, exist_ok=True)
  if dest.exists() and dest.stat().st_size >= min_bytes and not force_redownload:
    return

  t_start = time.time()
  task_id = progress.add_task(f"  Downloading {dest.name}", total=None)

  if use_aria2:
    unit_mult = {'B': 1, 'KiB': 1024, 'MiB': 1024**2, 'GiB': 1024**3, 'TiB': 1024**4}
    prog_re = re.compile(r'(\d+(?:\.\d+)?)(B|KiB|MiB|GiB|TiB)/(\d+(?:\.\d+)?)(B|KiB|MiB|GiB|TiB)')
    cmd = [
        'aria2c', '-x', '16', '-s', '16', '-k', '1M',
        '--allow-overwrite=true', '--summary-interval=1',
        '--download-result=hide', '--console-log-level=notice',
        '-d', str(dest.parent), '-o', dest.name, url,
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
      m = prog_re.search(line)
      if m:
        done = int(float(m.group(1)) * unit_mult[m.group(2)])
        total = int(float(m.group(3)) * unit_mult[m.group(4)])
        # Keep task open until aria2c exits so rounded GiB values don't finish it early
        progress.update(task_id, total=total, completed=min(done, max(total - 1, 0)))
    rc = proc.wait()
    if rc != 0 or not dest.exists() or dest.stat().st_size < min_bytes:
      raise RuntimeError(f"aria2c failed for {url}")
    size = dest.stat().st_size
    progress.update(task_id, total=size, completed=size)
    progress._tasks[task_id].finished_speed = size / max(time.time() - t_start, 1e-3)
  else:
    tmp = dest.with_suffix(dest.suffix + '.part')
    def _hook(blocks, block_size, total_size):
      if total_size > 0:
        progress.update(task_id, total=total_size, completed=min(blocks * block_size, max(total_size - 1, 0)))
    urllib.request.urlretrieve(url, tmp, reporthook=_hook)
    if tmp.stat().st_size < min_bytes:
      tmp.unlink(missing_ok=True)
      raise RuntimeError(f"Incomplete download from {url}")
    tmp.replace(dest)
    size = dest.stat().st_size
    progress.update(task_id, total=size, completed=size)
    progress._tasks[task_id].finished_speed = size / max(time.time() - t_start, 1e-3)


def _download_foundry_checkpoints_fast(needed: list[str], ckpt_dir: pathlib.Path, progress: Progress):
  """Resolve Foundry checkpoint URLs and save them under Foundry's canonical filenames via aria2c."""
  canonical_names = {
      'rfd3': 'rfd3_latest.ckpt',
      'rf3': 'rf3_foundry_01_24_latest_remapped.pt',
  }
  lookup = (
      "import json, pathlib, re, foundry; "
      "pkg = pathlib.Path(foundry.__file__).parent; "
      "urls = set(); "
      "[urls.update(re.findall(r'https?://[^\\s\"\\x27<>]+\\.(?:ckpt|pt)', p.read_text(errors='ignore'))) "
      " for p in pkg.rglob('*') if p.suffix in ('.py', '.json', '.yaml', '.yml')]; "
      "print(json.dumps(sorted(urls)))"
  )
  res = subprocess.run(
      [str(pathlib.Path(ENVS_DIR) / 'foundry/bin/python'), '-c', lookup],
      capture_output=True, text=True,
  )
  all_urls = []
  if res.returncode == 0 and res.stdout.strip():
    try:
      all_urls = json.loads(res.stdout.strip().splitlines()[-1])
    except Exception:
      all_urls = []

  fallback = []
  for name in needed:
    matches = [u for u in all_urls if f"/{name}_" in u.lower() or f"/{name}." in u.lower()]
    url = next((u for u in matches if 'latest' in u.lower()), matches[0] if matches else None)
    dest_name = canonical_names.get(name, url.split('/')[-1].split('?')[0] if url else None)
    if url and dest_name:
      dest = ckpt_dir / dest_name
      try:
        _download_with_rich(url, dest, progress, use_aria2=True, min_bytes=10_000_000)
      except Exception:
        fallback.append(name)
    else:
      fallback.append(name)

  if fallback:
    run_command_with_env('foundry', ['foundry', 'install', *fallback, '--checkpoint-dir', str(ckpt_dir)], quiet=True)


# 1. Foundry (RFD3 + built-in MPNN + optional RF3)
def setup_foundry(progress: Progress):
  t0 = time.time()
  env_path = pathlib.Path(ENVS_DIR) / 'foundry'
  ckpt_dir = pathlib.Path(FOUNDRY_CKPTS)

  if not env_path.exists() or force_reinstall:
    extra = 'all' if USE_RF3 else 'rfd3,mpnn'
    subprocess.run(['uv', 'venv', '-q', str(env_path), '--python', '3.12'], check=True)
    subprocess.run(
        ['uv', 'pip', 'install', '-q', '--python', str(env_path),
         f'rc-foundry[{extra}]', 'atomworks[ml]<3', '-U'],
        check=True,
    )
    site_custom = env_path / 'lib/python3.12/site-packages/sitecustomize.py'
    site_custom.write_text("import logging\nlogging.getLogger('atomworks.io').setLevel(logging.ERROR)\n")

  if force_redownload and ckpt_dir.exists():
    shutil.rmtree(ckpt_dir)
  ckpt_dir.mkdir(parents=True, exist_ok=True)

  needed = ['rfd3'] + (['rf3'] if USE_RF3 else [])
  expected = ['rfd3_latest.ckpt'] + (['rf3_foundry_01_24_latest_remapped.pt'] if USE_RF3 else [])
  if not all((ckpt_dir / f).exists() for f in expected):
    _download_foundry_checkpoints_fast(needed, ckpt_dir, progress)

  mpnn_weights = {
      # ProteinMPNN (48 nearest neighbors; σ = 0.02, 0.10, 0.20, 0.30 Å)
      'proteinmpnn_v_48_002.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/proteinmpnn_v_48_002.pt',
      'proteinmpnn_v_48_010.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/proteinmpnn_v_48_010.pt',
      'proteinmpnn_v_48_020.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/proteinmpnn_v_48_020.pt',
      'proteinmpnn_v_48_030.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/proteinmpnn_v_48_030.pt',
      # LigandMPNN (32 nearest neighbors, 25 ligand atom context; σ = 0.05, 0.10, 0.20, 0.30 Å)
      'ligandmpnn_v_32_005_25.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_005_25.pt',
      'ligandmpnn_v_32_010_25.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_010_25.pt',
      'ligandmpnn_v_32_020_25.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_020_25.pt',
      'ligandmpnn_v_32_030_25.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_030_25.pt',
      # SolubleMPNN (48 nearest neighbors; σ = 0.02, 0.10, 0.20, 0.30 Å)
      'solublempnn_v_48_002.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/solublempnn_v_48_002.pt',
      'solublempnn_v_48_010.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/solublempnn_v_48_010.pt',
      'solublempnn_v_48_020.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/solublempnn_v_48_020.pt',
      'solublempnn_v_48_030.pt': 'https://files.ipd.uw.edu/pub/ligandmpnn/solublempnn_v_48_030.pt',
  }

  for fname, url in mpnn_weights.items():
    _download_with_rich(url, ckpt_dir / fname, progress, use_aria2=False, min_bytes=100_000)

  return f"[bold green]✓ Foundry ready ({time.time() - t0:.0f} s)[/bold green] — checkpoints in [magenta]{ckpt_dir}[/magenta]"


# 1b. HBPLUS (for RFD3 hydrogen-bond conditioning & metrics)
def setup_hbplus(progress: Progress):
  t0 = time.time()
  dest = pathlib.Path(HBPLUS_BIN)
  if dest.exists() and os.access(dest, os.X_OK) and not force_reinstall:
    return f"[bold green]✓ HBPLUS ready (cached)[/bold green] — binary at [magenta]{dest}[/magenta]"

  with tempfile.TemporaryDirectory() as tmpdir:
    tmp = pathlib.Path(tmpdir)
    repo_dir = tmp / 'HBPLUS'
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-q', 'https://github.com/RomanLas/HBPLUS.git', str(repo_dir)],
        check=True,
    )
    tarball = repo_dir / 'hbplus.tar.gz'
    subprocess.run(['tar', '-xzf', str(tarball), '-C', str(tmp)], check=True)

    src_dir = next(tmp.rglob('hbp_main.c')).parent
    c_files = sorted(str(p) for p in src_dir.glob('*.c'))
    subprocess.run(['gcc', '-O2', '-std=gnu89', '-w', '-c', *c_files], cwd=src_dir, check=True)
    subprocess.run(
        ['gcc', '-o', 'hbplus', 'hbp_gen.o', 'hbp_inpdb.o', 'hbp_findh.o', 'hbp_hhb.o', 'hbp_qnh.o', 'hbp_main.o', '-lm'],
        cwd=src_dir,
        check=True,
    )
    shutil.copy2(src_dir / 'hbplus', dest)
    dest.chmod(0o755)

  return f"[bold green]✓ HBPLUS ready ({time.time() - t0:.0f} s)[/bold green] — binary at [magenta]{dest}[/magenta]"


# 2. Boltz-2 (optional)
def setup_boltz(progress: Progress):
  t0 = time.time()
  env_path = pathlib.Path(ENVS_DIR) / 'boltz'
  boltz_cache = pathlib.Path('/content/boltz/cache')
  os.environ['BOLTZ_CACHE'] = str(boltz_cache)

  if not env_path.exists() or force_reinstall:
    subprocess.run(['uv', 'venv', '-q', str(env_path), '--python', '3.10'], check=True)
    subprocess.run(['uv', 'pip', 'install', '-q', '--python', str(env_path), 'boltz[cuda]', '-U'], check=True)

  if force_redownload and boltz_cache.exists():
    shutil.rmtree(boltz_cache)
  boltz_cache.mkdir(parents=True, exist_ok=True)

  expected = ['mols.tar', 'boltz2_conf.ckpt', 'boltz2_aff.ckpt']
  if not all((boltz_cache / f).exists() for f in expected):
    dummy = pathlib.Path('/tmp/_boltz_prefetch.fasta')
    dummy.write_text(">A|protein|empty\nMAA\n")
    run_command_with_env(
        'boltz',
        ['boltz', 'predict', str(dummy), '--cache', str(boltz_cache), '--out_dir', '/tmp/_boltz_tmp'],
        quiet=True, check=False,
    )
    shutil.rmtree('/tmp/_boltz_tmp', ignore_errors=True)
    dummy.unlink(missing_ok=True)

  return f"[bold green]✓ Boltz-2 ready ({time.time() - t0:.0f} s)[/bold green] — cache in [magenta]{boltz_cache}[/magenta]"


# 3. AlphaFold3 / OpenFold3 (optional)
def setup_af3_stack(progress: Progress):
  t0 = time.time()
  af3_dir = pathlib.Path(AF3_DIR)
  af3_dir.mkdir(parents=True, exist_ok=True)
  env_path = pathlib.Path(ENVS_DIR) / 'af3'
  ready_stamp = af3_dir / f'ALPHAFOLD3_READY_{AF3_VERSION}'

  af3_model = 'alphafold3' if USE_AF3 else 'openfold3'
  af3_precision = 'fp32' if USE_AF3 else 'int8'
  af3_weights_dir = af3_dir / ('af3_native_weights' if USE_AF3 else 'weights')
  weights_stamp = af3_dir / f'WEIGHTS_DONE_{af3_model}_{af3_precision}'

  os.environ.update({
      'AF3_DIR': str(af3_dir),
      'AF3_MODEL': af3_model,
      'AF3_PRECISION': af3_precision,
      'AF3_WEIGHTS_DIR': str(af3_weights_dir),
  })

  if not env_path.exists() or not ready_stamp.exists() or force_reinstall:
    subprocess.run(['uv', 'venv', '-q', '--system-site-packages', str(env_path), '--python', sys.executable], check=True)
    colabfold_src = os.environ.get('RGI_COLABFOLD_SOURCE', 'git+https://github.com/sokrypton/ColabFold')
    subprocess.run(['uv', 'pip', 'install', '-q', '--python', str(env_path), '--no-deps', colabfold_src], check=True)
    subprocess.run(
        ['uv', 'pip', 'install', '-q', '--python', str(env_path),
         'dm-haiku==0.0.17', 'rdkit==2025.9.4', 'tokamax==0.0.11', 'ml_collections',
         'git+https://github.com/sokrypton/py2Dmol.git'],
        check=True,
    )
    subprocess.run(['uv', 'pip', 'uninstall', '-q', '--python', str(env_path), 'jax', 'jaxlib'], check=False)

    for attempt in range(4):
      res = subprocess.run(
          ['uv', 'pip', 'install', '-q', '--python', str(env_path), '--no-deps',
           f'alphafold3-colabfold=={AF3_VERSION}']
      )
      if res.returncode == 0:
        break
      time.sleep(20)
    else:
      raise RuntimeError(f"Could not install alphafold3-colabfold=={AF3_VERSION}")

    _download_with_rich(
        f'https://raw.githubusercontent.com/sokrypton/alphafold3/v{AF3_VERSION}/run_alphafold.py',
        af3_dir / 'run_alphafold.py',
        progress,
    )

    init_code = (
        "import os, pathlib, importlib.metadata as md, haiku, tokamax, alphafold3; "
        "from alphafold3.constants import ccd_fetch; "
        "jp = pathlib.Path(haiku.__file__).parent / '_src' / 'jaxpr_info.py'; "
        "jp.write_text(jp.read_text().replace('jax.core.DropVar', 'jax.extend.core.DropVar')) "
        "if jp.exists() and 'jax.core.DropVar' in jp.read_text() else None; "
        "gu = pathlib.Path(tokamax.__file__).parent / '_src' / 'gpu_utils.py'; "
        "old = 'return float(device.compute_capability) >= 8.0'; "
        "new = 'cc = float(device.compute_capability)\\n  return cc == 8.0 or cc >= 9.0'; "
        "gu.write_text(gu.read_text().replace(old, new)) if gu.exists() and old in gu.read_text() else None; "
        "root = os.path.dirname(md.distribution('alphafold3-colabfold').locate_file('alphafold3')); "
        "conv = os.path.join(root, 'alphafold3', 'constants', 'converters'); "
        "os.makedirs(conv, exist_ok=True); "
        "ccd_fetch.write_pickles(ccd_fetch.codes_for_input(), os.path.join(conv, 'ccd.pickle'), "
        "os.path.join(conv, 'chemical_component_sets.pickle'), libcifpp_dir=os.path.join(root, 'share', 'libcifpp'))"
    )
    run_command_with_env('af3', ['python', '-c', init_code], quiet=True)
    ready_stamp.touch()

  if force_redownload:
    shutil.rmtree(af3_weights_dir, ignore_errors=True)
    weights_stamp.unlink(missing_ok=True)
  af3_weights_dir.mkdir(parents=True, exist_ok=True)

  if USE_AF3:
    af3_bin = af3_weights_dir / 'af3.bin.zst'
    if not glob.glob(str(af3_weights_dir / '*.bin.zst')):
      _download_with_rich(AF3_WEIGHTS_URL, af3_bin, progress, use_aria2=True, min_bytes=1_000_000)
    weights_stamp.touch()
  elif not weights_stamp.exists():
    run_command_with_env(
        'af3',
        ['python', '-c', f"from alphafold3.model import weights; weights.ensure_weights('{af3_model}', None, precision='{af3_precision}')"],
        quiet=True,
    )
    weights_stamp.touch()

  return f"[bold green]✓ {folding_model} ready ({time.time() - t0:.0f} s)[/bold green] — weights in [magenta]{af3_weights_dir}[/magenta]"


# Header -> Progress bars -> Summary -> GPU test (clean top-to-bottom layout)
tasks = [setup_foundry, setup_hbplus]
if USE_BOLTZ:
  tasks.append(setup_boltz)
if USE_AF3_STACK:
  tasks.append(setup_af3_stack)

console.print(f"[bold blue]Installing Foundry[/bold blue] (RFD3 + MPNN + HBPLUS) and [bold blue]{folding_model}[/bold blue] in parallel...")

summary = []
with Progress(
    TextColumn("{task.description}"),
    BarColumn(),
    DownloadColumn(),
    TransferSpeedColumn(),
    TimeRemainingColumn(elapsed_when_finished=True),
    console=console,
) as progress:
  with ThreadPoolExecutor(max_workers=len(tasks)) as pool:
    futures = [pool.submit(fn, progress) for fn in tasks]
    for fut in as_completed(futures):
      summary.append(fut.result())

for line in summary:
  if line:
    console.print(line)

# Verify GPU access and HBPLUS across active environments
console.print("\n[bold]========== TEST GPU-CUDA & HBPLUS ==========[/bold]")
torch_test = "import torch; print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())"
run_command_with_env("foundry", ["python", "-c", torch_test])
hb_test = (
    "import os, subprocess; "
    "p = os.environ.get('HBPLUS_PATH'); "
    "assert p and os.access(p, os.X_OK), f'Invalid HBPLUS_PATH: {p}'; "
    "out = subprocess.run([p], capture_output=True, text=True); "
    "print('HBPLUS_PATH:', p, '| Banner:', (out.stdout or out.stderr).splitlines()[0])"
)
run_command_with_env("foundry", ["python", "-c", hb_test])
if USE_BOLTZ:
  run_command_with_env("boltz", ["python", "-c", torch_test])
if USE_AF3_STACK:
  jax_test = "import jax, alphafold3; print('JAX:', jax.__version__, 'Devices:', jax.devices())"
  run_command_with_env("af3", ["python", "-c", jax_test])

# ===========================
#@markdown VARS that are set and used later
#@markdown | Variable | Value | Set |
#@markdown |---|---|---|
#@markdown | `FOLDING_MODEL` | `"RosettaFold3"` / `"Boltz2"` / `"AlphaFold3"` / `"OpenFold3"` | always |
#@markdown | `USE_BOLTZ`, `USE_RF3`, `USE_AF3`, `USE_OF3` | `"True"` / `"False"` (string) | always |
#@markdown | `ENVS_DIR` | `/content/envs` | always |
#@markdown | `FOUNDRY_CHECKPOINT_DIRS` | `/content/foundry/checkpoints` or `/content/drive/MyDrive/foundry_checkpoints` | always |
#@markdown | `HBPLUS_PATH` | `/usr/local/bin/hbplus` | always |
#@markdown | `BOLTZ_CACHE` | `/content/boltz/cache` | only if `USE_BOLTZ` |
#@markdown | `AF3_DIR` | `/content/af3` (contains `run_alphafold.py`) | only if `USE_AF3` or `USE_OF3` |
#@markdown | `AF3_MODEL` | `"alphafold3"` / `"openfold3"` | only if `USE_AF3` or `USE_OF3` |
#@markdown | `AF3_PRECISION` | `"fp32"` (AF3) / `"int8"` (OpenFold3) | only if `USE_AF3` or `USE_OF3` |
#@markdown | `AF3_WEIGHTS_DIR` | `/content/af3/af3_native_weights` or `/content/af3/weights` | only if `USE_AF3` or `USE_OF3` |
#@markdown | `AF3_CACHE_DIR` | `/tmp/af3_cache` or `/content/drive/MyDrive/.af3_cache` | only if `USE_AF3` or `USE_OF3` |

In [ ]:
#@title Helper functions to compare RFD3 designs vs structure predictions (Glycoprotein Binders)

try:
  import biotite
except ModuleNotFoundError:
  !pip -q install biotite

import glob, gzip, io, json, pathlib, re, uuid
import numpy as np
import biotite.structure as struc
import biotite.structure.io.pdb as pdb_io
import biotite.structure.io.pdbx as pdbx_io
from IPython.display import display, HTML

COLORS = ["#F58518", "#54A24B", "#E45756", "#B279A2", "#72B7B2", "#EECA3B", "#FF9DA6", "#4C78A8"]
CHAIN_LABELS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789")


def _strip_struct_ext(path: pathlib.Path) -> str:
  name = path.name
  for ext in (".gz", ".cif", ".pdb", ".bcif"):
    name = name.removesuffix(ext)
  return name


def _load(path):
  p = str(path)
  opener = gzip.open if p.endswith(".gz") else open
  base = p[:-3] if p.endswith(".gz") else p
  if base.endswith(".bcif"):
    with opener(p, "rb") as f:
      return pdbx_io.get_structure(pdbx_io.BinaryCIFFile.read(f), model=1, extra_fields=["b_factor"])
  if base.endswith(".cif"):
    with opener(p, "rt") as f:
      return pdbx_io.get_structure(pdbx_io.CIFFile.read(f), model=1, extra_fields=["b_factor"])
  with opener(p, "rt") as f:
    return pdb_io.PDBFile.read(f).get_structure(model=1, extra_fields=["b_factor"])


def _atom_elements(struct):
  """Return uppercase stripped element symbols, falling back to atom_name if missing."""
  elems = (
      np.char.strip(np.char.upper(struct.element.astype(str)))
      if hasattr(struct, "element")
      else np.full(len(struct), "", dtype=object)
  )
  missing = np.isin(elems, ["", "?", "."])
  if np.any(missing):
    is_prot = struc.filter_amino_acids(struct)
    names = np.char.strip(np.char.upper(struct.atom_name.astype(str)))
    two_letter = {"CL", "BR", "NA", "MG", "ZN", "FE", "CA", "MN", "CU", "CO", "NI", "SE"}
    elems = elems.astype(object)
    for idx in np.where(missing)[0]:
      m = re.match(r"^[0-9]*([A-Z]+)", str(names[idx]))
      if not m:
        elems[idx] = ""
      else:
        letters = m.group(1)
        elems[idx] = letters[:2] if (not is_prot[idx] and letters[:2] in two_letter) else letters[0]
    elems = elems.astype(str)
  return elems


def _ligand_mask(struct):
  """Select non-polymer, non-solvent heavy atoms (captures glycans, small molecules, and ions)."""
  is_polymer = struc.filter_amino_acids(struct) | struc.filter_nucleotides(struct)
  return (~is_polymer) & (~struc.filter_solvent(struct)) & (_atom_elements(struct) != "H")


def _to_pdb(atoms):
  """Serialize protein + glycan/ligand/ion heavy atoms to PDB with explicit HETATM flags."""
  is_prot = struc.filter_amino_acids(atoms)
  is_lig = _ligand_mask(atoms)
  keep = atoms[(is_prot | is_lig) & (_atom_elements(atoms) != "H")].copy()
  keep.hetero = ~struc.filter_amino_acids(keep)

  # Preserve standard 1-char chain IDs (A, B, C...); only remap multi-char IDs if present
  raw_cids = keep.chain_id.astype(str)
  used = {c for c in raw_cids if len(c) == 1}
  avail = [c for c in CHAIN_LABELS if c not in used]
  cid_map = {}
  for c in dict.fromkeys(raw_cids):
    if len(c) == 1:
      cid_map[c] = c
    else:
      cid_map[c] = avail.pop(0) if avail else c[:1]
  keep.chain_id = np.array([cid_map[c] for c in raw_cids])
  keep.res_name = np.array([str(r)[:3] for r in keep.res_name])
  keep.atom_name = np.array([str(a)[:4] for a in keep.atom_name])

  buf = io.StringIO()
  f = pdb_io.PDBFile()
  f.set_structure(keep)
  f.write(buf)
  return buf.getvalue()


def _superimpose_ca(fixed_ca, mobile_ca):
  if len(fixed_ca) == len(mobile_ca) and len(fixed_ca) >= 3:
    res = struc.superimpose(fixed_ca, mobile_ca)
    tf = res[1] if isinstance(res, tuple) else res
    rmsd = float(struc.rmsd(fixed_ca, tf.apply(mobile_ca)))
    return tf, rmsd, len(fixed_ca)
  res = struc.superimpose_structural_homologs(fixed_ca, mobile_ca)
  tf, f_idx, m_idx = res[1:] if len(res) == 4 else res
  rmsd = float(struc.rmsd(fixed_ca[f_idx], tf.apply(mobile_ca[m_idx])))
  return tf, rmsd, len(f_idx)


def _ligand_rmsd(fixed, aligned_mobile):
  """Pooled heavy-atom RMSD across all glycan/ligand residues after target alignment."""
  def _get_het_residues(struct):
    het = struct[_ligand_mask(struct)]
    if len(het) == 0:
      return []
    residues = []
    for chain_id in dict.fromkeys(het.chain_id.astype(str)):
      ch = het[het.chain_id.astype(str) == chain_id]
      for res_id in dict.fromkeys(ch.res_id):
        res = ch[ch.res_id == res_id]
        if len(res) >= 3:
          rname = str(res.res_name[0]).strip().upper()
          centroid = np.mean(res.coord, axis=0)
          amap = {str(a.atom_name).strip(): a.coord for a in res}
          residues.append((rname, centroid, amap))
    return residues

  f_res = _get_het_residues(fixed)
  m_res = _get_het_residues(aligned_mobile)
  if not f_res or not m_res:
    return None

  # 1-to-1 greedy nearest-centroid matching (prioritizing identical residue names)
  pairs = []
  for i, (fn, fc, _) in enumerate(f_res):
    for j, (mn, mc, _) in enumerate(m_res):
      dist = float(np.linalg.norm(fc - mc))
      pairs.append((0 if fn == mn else 1, dist, i, j))
  pairs.sort(key=lambda x: (x[0], x[1]))

  used_f, used_m, matched = set(), set(), []
  for _, dist, i, j in pairs:
    if i not in used_f and j not in used_m:
      used_f.add(i)
      used_m.add(j)
      matched.append((i, j, dist))

  if not matched:
    return None

  sq_dists = []
  for i, j, _ in matched:
    f_map, m_map = f_res[i][2], m_res[j][2]
    common = sorted(set(f_map) & set(m_map))
    if len(common) >= 3:
      fc = np.array([f_map[k] for k in common])
      mc = np.array([m_map[k] for k in common])
      sq_dists.append(np.sum((fc - mc) ** 2, axis=-1))

  label = f"Glycans ({len(matched)})" if len(matched) > 1 else "Ligand"
  if sq_dists:
    rmsd = float(np.sqrt(np.mean(np.concatenate(sq_dists))))
    return f"{label}: {rmsd:.2f} Å"

  com_rmsd = float(np.sqrt(np.mean([d ** 2 for _, _, d in matched])))
  return f"{label} COM: {com_rmsd:.2f} Å"


def _align(fixed, fixed_ca, mobile, align_chain="auto", binder_chain="A"):
  mobile_ca = mobile[(mobile.atom_name == "CA") & struc.filter_amino_acids(mobile)]
  f_chains = set(fixed_ca.chain_id.astype(str))
  m_chains = set(mobile_ca.chain_id.astype(str))

  target_chain = "B" if binder_chain == "A" else "A"
  use_target = (align_chain == target_chain) or (
      align_chain == "auto" and {binder_chain, target_chain}.issubset(f_chains) and {binder_chain, target_chain}.issubset(m_chains)
  )

  if use_target and target_chain in f_chains and target_chain in m_chains:
    f_t = fixed_ca[fixed_ca.chain_id.astype(str) == target_chain]
    m_t = mobile_ca[mobile_ca.chain_id.astype(str) == target_chain]
    tf, t_rmsd, _ = _superimpose_ca(f_t, m_t)
    aligned_mobile = tf.apply(mobile)

    parts = [f"Target {target_chain}: {t_rmsd:.2f} Å"]
    if binder_chain in f_chains and binder_chain in m_chains:
      f_a = fixed_ca[fixed_ca.chain_id.astype(str) == binder_chain]
      m_a_pose = aligned_mobile[
          (aligned_mobile.atom_name == "CA")
          & struc.filter_amino_acids(aligned_mobile)
          & (aligned_mobile.chain_id.astype(str) == binder_chain)
      ]
      if len(f_a) == len(m_a_pose):
        pose_rmsd = float(struc.rmsd(f_a, m_a_pose))
        parts.append(f"Binder pose: {pose_rmsd:.2f} Å")
      _, fold_rmsd, _ = _superimpose_ca(f_a, mobile_ca[mobile_ca.chain_id.astype(str) == binder_chain])
      parts.append(f"Binder fold: {fold_rmsd:.2f} Å")

    lig_note = _ligand_rmsd(fixed, aligned_mobile)
    if lig_note:
      parts.append(lig_note)

    return _to_pdb(aligned_mobile), " | ".join(parts)

  tf, rmsd, n_res = _superimpose_ca(fixed_ca, mobile_ca)
  aligned_mobile = tf.apply(mobile)
  lig_note = _ligand_rmsd(fixed, aligned_mobile)
  note = f"{rmsd:.2f} Å ({n_res} res)" + (f" | {lig_note}" if lig_note else "")
  return _to_pdb(aligned_mobile), note


def browse_designs(
    design_glob,
    pred_root,
    only_best=True,
    max_models=5,
    width=700,
    height=500,
    align_chain="auto",
    binder_chain="A",
):
  """Interactive viewer to compare RFD3 glycoprotein binder designs against AF3 / OpenFold3 / Boltz-2 / RF3 models."""
  pred_root = pathlib.Path(pred_root)
  payload = []

  for design_path in sorted(glob.glob(str(design_glob))):
    stem = _strip_struct_ext(pathlib.Path(design_path))

    candidates = (
        pred_root / stem, pred_root / stem.lower(),
        pred_root / f"{stem}.cif", pred_root / f"{stem}.cif".lower(),
        pred_root / "predictions" / stem, pred_root / "predictions" / f"{stem}.cif",
    )
    design_dir = next((d for d in candidates if d.is_dir()), None)
    if design_dir is not None:
      matches = [
          p for pat in ("*.cif", "*.cif.gz", "*.pdb")
          for p in sorted(design_dir.rglob(pat))
          if p.is_file() and "inputs" not in p.parts
      ]
    else:
      matches = [
          p for pat in (f"{stem}*model*.cif", f"{stem}*model*.cif.gz", f"{stem}*model*.pdb")
          for p in sorted(pred_root.rglob(pat))
          if p.is_file() and "inputs" not in p.parts
      ]

    if only_best:
      top_dir = design_dir if design_dir is not None else pred_root
      top_af3_rf3 = [m for m in matches if m.parent == top_dir and m.name.endswith(("_model.cif", "_model.cif.gz"))]
      top_boltz = [m for m in matches if m.name.endswith(("_model_0.cif", "_model_0.pdb"))]
      matches = (top_af3_rf3 or top_boltz or matches)[:1]
    else:
      samples = [m for m in matches if "sample-" in str(m) or "_model_" in m.name]
      matches = (samples or matches)[:max_models]

    if not matches:
      continue

    fixed = _load(design_path)
    fixed_ca = fixed[(fixed.atom_name == "CA") & struc.filter_amino_acids(fixed)]
    models = []
    for i, m in enumerate(matches):
      pdb_str, note = _align(fixed, fixed_ca, _load(m), align_chain=align_chain, binder_chain=binder_chain)
      models.append({
          "label": m.parent.name if m.name == "model.cif" else _strip_struct_ext(m),
          "color": COLORS[i % len(COLORS)],
          "pdb": pdb_str,
          "note": note,
      })
    payload.append({"stem": stem, "fixed_pdb": _to_pdb(fixed), "models": models})

  if not payload:
    raise FileNotFoundError("No matching design/prediction pairs found. Check design_glob and pred_root.")

  uid = uuid.uuid4().hex[:8]
  data_json = json.dumps(payload).replace("</", "<\\/")
  binder_js = json.dumps(str(binder_chain))

  display(HTML(f"""
  <div style="font-family:sans-serif;max-width:{width}px;">
    <div style="display:flex;gap:6px;align-items:center;margin-bottom:8px;">
      <button id="prev_{uid}" style="padding:5px 12px;cursor:pointer;">◀ Prev</button>
      <select id="sel_{uid}" style="flex:1;padding:5px;"></select>
      <button id="next_{uid}" style="padding:5px 12px;cursor:pointer;">Next ▶</button>
    </div>
    <div id="view_{uid}" style="position:relative;width:{width}px;height:{height}px;border:1px solid #ddd;"></div>
    <div id="leg_{uid}" style="font-size:12px;color:#444;margin-top:6px;line-height:1.6;"></div>
  </div>
  <script>
  (function() {{
    const DATA = {data_json};
    const BINDER_CHAIN = {binder_js};
    const TARGET_COLOR = "#4C78A8";
    const BINDER_COLOR = "#F58518";
    const LIGAND_COLOR = "#54A24B";
    const METALS = ["ZN", "MG", "CA", "FE", "MN", "CU", "CO", "NI"];
    const sel = document.getElementById("sel_{uid}");
    const leg = document.getElementById("leg_{uid}");
    const esc = s => String(s).replace(/[&<>"']/g, c => ({{"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}}[c]));
    const swatch = (c, t, m) => `<span style="display:inline-block;width:10px;height:10px;background:${{c}};border:1px solid #888;margin:0 4px 0 ${{m}}px;vertical-align:middle;"></span>${{esc(t)}}`;

    function init() {{
      const v = $3Dmol.createViewer(document.getElementById("view_{uid}"), {{ backgroundColor: "white" }});
      DATA.forEach((d, i) => sel.add(new Option(`${{i + 1}}/${{DATA.length}} — ${{d.stem}} (${{d.models.length}} model${{d.models.length > 1 ? "s" : ""}})`, i)));

      function render() {{
        const d = DATA[Number(sel.value)];
        v.removeAllModels();

        // RFD3 reference: light grey target + slightly darker grey binder + grayCarbon glycans/ligand
        v.addModel(d.fixed_pdb, "pdb");
        v.setStyle({{ model: 0, hetflag: false }}, {{ cartoon: {{ color: "gainsboro", opacity: 0.70 }} }});
        v.setStyle({{ model: 0, chain: BINDER_CHAIN, hetflag: false }}, {{ cartoon: {{ color: "#BDBDBD", opacity: 0.75 }} }});
        v.setStyle({{ model: 0, hetflag: true }}, {{ stick: {{ colorscheme: "grayCarbon", radius: 0.22 }} }});
        v.addStyle({{ model: 0, hetflag: true, elem: METALS }}, {{ sphere: {{ color: "gray", radius: 0.55 }} }});

        let html = swatch("gainsboro", `${{d.stem}} (RFD3)`, 0);
        const singleModel = d.models.length === 1;

        d.models.forEach((m, i) => {{
          const mdl = v.addModel(m.pdb, "pdb");
          if (singleModel) {{
            // Single predicted model: Target (blue), Designed Binder (orange), Glycans/Ligand (greenCarbon)
            v.setStyle({{ model: i + 1, hetflag: false }}, {{ cartoon: {{ color: TARGET_COLOR }} }});
            v.setStyle({{ model: i + 1, chain: BINDER_CHAIN, hetflag: false }}, {{ cartoon: {{ color: BINDER_COLOR }} }});
            v.setStyle({{ model: i + 1, hetflag: true }}, {{ stick: {{ colorscheme: "greenCarbon", radius: 0.20 }} }});
            v.addStyle({{ model: i + 1, hetflag: true, elem: METALS }}, {{ sphere: {{ color: LIGAND_COLOR, radius: 0.50 }} }});

            html += `<span style="margin-left:12px;"><b>${{esc(m.label)}}</b> (${{esc(m.note)}})</span>`;
            html += swatch(TARGET_COLOR, "Target", 8);
            html += swatch(BINDER_COLOR, `Binder (${{BINDER_CHAIN}})`, 8);
            if (mdl.selectedAtoms({{ hetflag: true }}).length > 0) {{
              html += swatch(LIGAND_COLOR, "Glycan / Ligand (pred)", 8);
            }}
          }} else {{
            // Multiple samples: muted target, distinct color per sample's binder & glycans
            v.setStyle({{ model: i + 1, hetflag: false }}, {{ cartoon: {{ color: TARGET_COLOR, opacity: 0.45 }} }});
            v.setStyle({{ model: i + 1, chain: BINDER_CHAIN, hetflag: false }}, {{ cartoon: {{ color: m.color }} }});
            v.setStyle({{ model: i + 1, hetflag: true }}, {{ stick: {{ color: m.color, radius: 0.20 }} }});
            v.addStyle({{ model: i + 1, hetflag: true, elem: METALS }}, {{ sphere: {{ color: m.color, radius: 0.50 }} }});
            html += swatch(m.color, `${{m.label}} (${{m.note}})`, 12);
          }}
        }});

        v.zoomTo();
        v.render();
        leg.innerHTML = html;
      }}

      sel.onchange = render;
      document.getElementById("prev_{uid}").onclick = () => {{ sel.value = (Number(sel.value) - 1 + DATA.length) % DATA.length; render(); }};
      document.getElementById("next_{uid}").onclick = () => {{ sel.value = (Number(sel.value) + 1) % DATA.length; render(); }};
      render();
    }}

    if (window.$3Dmol) init();
    else {{ const s = document.createElement("script"); s.src = "https://3Dmol.org/build/3Dmol-min.js"; s.onload = init; document.head.appendChild(s); }}
  }})();
  </script>
  """))

# Usage:
# browse_designs("glycan_binder_design/rfd3_out/*.cif.gz", "glycan_binder_design/alphafold3_out", only_best=True)

## Target Preparation: Glycoprotein Binder Design (`8S6Z` — CD28 + Glycans)

In this exercise, we design *de novo* binders against **CD28** (`8S6Z`) while **explicitly modeling its N-linked `NAG` glycans** so RFD3 avoids steric clashes with the glycan shield and respects the glycosylated surface (following the [Foundry Binder Design Tutorial](https://rosettacommons.github.io/foundry/models/rfd3/tutorials/binder_design_tutorial.html#binder-design-glycan-example)).

| Cell / Parameter | Glycoprotein Exercise (`8S6Z` — CD28 + `NAG`) |
|---|---|
| **Goal** | Design a *de novo* binder to glycosylated CD28 (replacing the AI3 antibody Fab) |
| **Cell 1 — `pdb_id`** | `8S6Z` |
| **Cell 2 — `target_chain`** | `C` *(or `F`; `8S6Z` contains two CD28 copies — check Cell 1 inventory)* |
| **Cell 2 — `glycan_resnames`** | `NAG` *(comma-separated for branched trees, e.g., `NAG, BMA, MAN, FUC`)* |
| **Cell 2 — `binder_chain`** | Reference Fab chains contacting your chosen CD28 copy *(inspect Cell 1 output, e.g., `A, B` for Chain `C`, or leave blank)* |
| **Cell 2 — `keep_ions`** | *(leave blank — drops `ZN` ions and solvent)* |
| **Cell 2 — `crop_range`** | *(leave blank — keep full CD28 ectodomain, 118 aa)* |
| **Cell 3 — `view_mode`** | Start with `Complex (with binder)` to inspect Fab hotspots, then switch to `Final Target` to verify Chain `B` + glycan chains `C–G` |

### What to Check in Each Step
1. **Cell 1 (Fetch & Inspect):** Identify the two CD28 chains (`C` and `F`), the antibody Fab chains, the `NAG` glycan residues, and any `ZN` ions/solvent to drop.
2. **Cell 2 (Clean, Relabel & Preserve `LINK`s):** Standardizes the glycoprotein to the Foundry RFD3 convention:
   - **Chain `B`** = cleaned, continuously renumbered CD28 (`B1–B118`)
   - **Chains `C, D, E, F, G, ...`** = each covalently attached glycan tree on its own chain (`C1-2, D1-1, E1-1, F1-1, G1`)
   - **`LINK` records** = preserved for every `ASN–NAG` and `NAG–NAG` covalent bond (and stored in `PREP_STATE` for refolding)
   - **Chain `A`** = reserved for the *de novo* binder generated by RFD3
3. **Cell 3 (Visualize & Inspect Hotspots):** Displays the target, all glycan chains (in green sticks), and prints target residues within `contact_cutoff` of the reference Fab using their **new Chain `B` residue numbers**.

In [ ]:
#@title 1. Fetch PDB & Inspect Chains, Glycans & Ligands
pdb_id = "8S6Z" #@param {type:"string"}
#@markdown Optional: path to a local `.pdb` or `.cif` file (leave blank to download `pdb_id` from RCSB)
local_file = "" #@param {type:"string"}
work_dir = "glycoprotein_binder_prep" #@param {type:"string"}

try:
  import Bio
except ModuleNotFoundError:
  !pip -q install biopython

try:
  import py3Dmol
except ModuleNotFoundError:
  !pip -q install py3Dmol
  import py3Dmol

import pathlib
from Bio.PDB import PDBList, PDBParser, MMCIFParser

WORK_DIR = pathlib.Path(work_dir).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

if local_file.strip():
  RAW_PATH = pathlib.Path(local_file.strip()).resolve()
  PDB_ID = RAW_PATH.stem.upper()
else:
  PDB_ID = pdb_id.strip().upper()
  pdbl = PDBList()
  RAW_PATH = pathlib.Path(
      pdbl.retrieve_pdb_file(PDB_ID, pdir=str(WORK_DIR), file_format="pdb", overwrite=False)
  ).resolve()

parser = MMCIFParser(QUIET=True) if RAW_PATH.suffix.lower() == ".cif" else PDBParser(QUIET=True)
model = parser.get_structure(PDB_ID, str(RAW_PATH))[0]

print(f"=== Inventory for {PDB_ID} ({RAW_PATH.name}) ===\n")
print("Protein Chains:")
for chain in model:
  aa_nums = [
      r.id[1] for r in chain
      if r.id[0] == " " and all(a in r for a in ("N", "CA", "C", "O"))
  ]
  if aa_nums:
    gaps = [f"{aa_nums[i]+1}-{aa_nums[i+1]-1}" for i in range(len(aa_nums)-1) if aa_nums[i+1] - aa_nums[i] > 1]
    gap_str = f" | unresolved gaps: {', '.join(gaps)}" if gaps else " | no internal gaps"
    print(f"  Chain {chain.id}: {len(aa_nums)} aa (res {aa_nums[0]}–{aa_nums[-1]}){gap_str}")

print("\nNon-Water Glycans, Ligands & Ions (HETATM):")
het_found = False
for chain in model:
  for r in chain:
    if r.id[0] != " " and r.id[0] != "W" and r.get_resname().strip() != "HOH":
      heavy = sum(1 for a in r.get_atoms() if a.element != "H")
      print(f"  {r.get_resname().strip():<4} | Chain {chain.id} | res {r.id[1]:<4} | {heavy} heavy atoms")
      het_found = True
if not het_found:
  print("  (none)")

In [ ]:
#@title 2. Clean Glycoprotein, Relabel Chains & Preserve Glycan LINK Records
target_chain = "C" #@param {type:"string"}
#@markdown Comma-separated glycan residue codes to keep (e.g., `NAG` or `NAG, BMA, MAN, FUC`):
glycan_resnames = "NAG" #@param {type:"string"}
#@markdown Reference binder chain(s) to inspect before removal (comma-separated, e.g., `A, B` for a Fab), or leave blank:
binder_chain = "A, B" #@param {type:"string"}
#@markdown Optional: comma-separated ion codes to keep after glycan chains (e.g., `ZN`), or leave blank to drop all ions:
keep_ions = "" #@param {type:"string"}
#@markdown Optional: crop target to original residue range (e.g., `18-131`), or leave blank to keep full chain:
crop_range = "" #@param {type:"string"}
#@markdown Maximum heavy-atom distance (Å) to detect covalent protein–glycan and glycan–glycan bonds:
covalent_cutoff = 2.0 #@param {type:"number"}

import collections
import numpy as np
from Bio.PDB import PDBParser, MMCIFParser, PDBIO
from Bio.PDB.Structure import Structure
from Bio.PDB.Model import Model
from Bio.PDB.Chain import Chain
from Bio.PDB.Residue import Residue

parser = MMCIFParser(QUIET=True) if RAW_PATH.suffix.lower() == ".cif" else PDBParser(QUIET=True)
orig_model = parser.get_structure(PDB_ID, str(RAW_PATH))[0]

t_chain_id = target_chain.strip()
b_chain_ids = [c.strip() for c in binder_chain.split(",") if c.strip()]
glycan_codes = {x.strip().upper() for x in glycan_resnames.split(",") if x.strip()}
ion_codes = {x.strip().upper() for x in keep_ions.split(",") if x.strip()}

crop_min, crop_max = None, None
if crop_range.strip():
  parts = crop_range.strip().split("-")
  crop_min, crop_max = int(parts[0]), int(parts[1])

def _copy_clean_residue(res, new_id):
  """Copy residue keeping only non-H atoms and primary altloc (' ' or 'A')."""
  new_res = Residue(new_id, res.get_resname(), res.segid)
  for atom in res.get_atoms():
    a = atom.selected_child if atom.is_disordered() else atom
    if a.element == "H":
      continue
    if atom.is_disordered():
      altlocs = {c.get_altloc(): c for c in atom.disordered_get_list()}
      a = altlocs.get("A", altlocs.get(" ", a))
    a_copy = a.copy()
    a_copy.set_altloc(" ")
    a_copy.disordered_flag = 0
    new_res.add(a_copy)
  return new_res

def _heavy_atoms(res):
  out = []
  for atom in res.get_atoms():
    a = atom.selected_child if atom.is_disordered() else atom
    if a.element != "H":
      out.append(a)
  return out

def _closest_atom_pair(res1, res2):
  best = (999.0, None, None)
  for a1 in _heavy_atoms(res1):
    for a2 in _heavy_atoms(res2):
      d = float(np.linalg.norm(a1.coord - a2.coord))
      if d < best[0]:
        best = (d, a1.get_name(), a2.get_name())
  return best

# 1. Extract target protein residues
if t_chain_id not in orig_model:
  raise ValueError(f"Target chain '{t_chain_id}' not found in {PDB_ID}.")

target_aa_orig = [
    r for r in orig_model[t_chain_id]
    if r.id[0] == " " and all(a in r for a in ("N", "CA", "C", "O"))
    and (crop_min is None or crop_min <= r.id[1] <= crop_max)
]

# 2. Identify all glycan residues covalently attached (directly or via sugar chain) to target_aa_orig
all_glycans = [
    (c.id, r) for c in orig_model for r in c
    if r.id[0] != " " and r.get_resname().strip().upper() in glycan_codes
]

# Build adjacency: target -> root glycans, and glycan -> glycan
root_links = {}  # gly_idx -> (target_res_orig_num, target_atom, gly_atom, dist)
for g_idx, (_, g_res) in enumerate(all_glycans):
  best = (covalent_cutoff + 1.0, None, None, None)
  for t_res in target_aa_orig:
    d, a_t, a_g = _closest_atom_pair(t_res, g_res)
    if d <= covalent_cutoff and d < best[0]:
      best = (d, t_res.id[1], a_t, a_g)
  if best[1] is not None:
    root_links[g_idx] = (best[1], best[2], best[3], best[0])

gly_adj = collections.defaultdict(list)  # g_i -> [(g_j, a_i, a_j, d)]
for i in range(len(all_glycans)):
  for j in range(i + 1, len(all_glycans)):
    d, a_i, a_j = _closest_atom_pair(all_glycans[i][1], all_glycans[j][1])
    if d <= covalent_cutoff:
      gly_adj[i].append((j, a_i, a_j, d))
      gly_adj[j].append((i, a_j, a_i, d))

# BFS from each root glycan (ordered by target residue number) to form discrete glycan trees
visited = set()
glycan_trees = []  # list of [g_idx, ...] in BFS order from attachment site
for root_idx in sorted(root_links, key=lambda idx: root_links[idx][0]):
  if root_idx in visited:
    continue
  tree, queue = [], collections.deque([root_idx])
  visited.add(root_idx)
  while queue:
    curr = queue.popleft()
    tree.append(curr)
    for nxt, _, _, _ in sorted(gly_adj[curr], key=lambda x: x[0]):
      if nxt not in visited:
        visited.add(nxt)
        queue.append(nxt)
  glycan_trees.append(tree)

kept_ions = [
    r for c in orig_model for r in c
    if r.id[0] != " " and r.get_resname().strip().upper() in ion_codes
]

# Helper to format standard PDB LINK records (columns per wwPDB format)
def _format_link_line(atom1, res1, ch1, num1, atom2, res2, ch2, num2, dist):
  a1 = f" {atom1:<3}" if len(atom1) < 4 else f"{atom1:<4}"
  a2 = f" {atom2:<3}" if len(atom2) < 4 else f"{atom2:<4}"
  return (
      f"LINK        {a1} {res1:>3} {ch1:1}{num1:>4}                "
      f"{a2} {res2:>3} {ch2:1}{num2:>4}     1555   1555 {dist:5.2f}\n"
  )

# 3. Save intermediate complex PDB (if reference binder chain(s) provided)
complex_pdb = None
if b_chain_ids:
  for bc in b_chain_ids:
    if bc not in orig_model:
      raise ValueError(f"Binder chain '{bc}' not found in {PDB_ID}.")
  c_struct = Structure(f"{PDB_ID}_complex")
  c_model = Model(0)
  c_struct.add(c_model)

  used_cids = set([t_chain_id] + b_chain_ids)
  for cid in [t_chain_id] + b_chain_ids:
    ch = Chain(cid)
    for r in orig_model[cid]:
      if r.id[0] == " " and all(a in r for a in ("N", "CA", "C", "O")):
        if cid == t_chain_id and crop_min is not None and not (crop_min <= r.id[1] <= crop_max):
          continue
        ch.add(_copy_clean_residue(r, r.id))
    c_model.add(ch)

  avail_cids = [c for c in "CDEFGHIJKLMNOPQRSTUVWXYZ" if c not in used_cids]
  for tree in glycan_trees:
    g_cid = avail_cids.pop(0)
    ch_g = Chain(g_cid)
    for r_num, g_idx in enumerate(tree, start=1):
      g_res = all_glycans[g_idx][1]
      g_code = g_res.get_resname().strip().upper()
      ch_g.add(_copy_clean_residue(g_res, (f"H_{g_code}", r_num, " ")))
    c_model.add(ch_g)

  complex_pdb = WORK_DIR / f"{PDB_ID}_complex.pdb"
  io = PDBIO()
  io.set_structure(c_struct)
  io.save(str(complex_pdb))

# 4. Save final RFD3-ready target PDB: Target -> Chain B (1..N), Glycan trees -> Chains C, D, E..., Ions -> next chain
f_struct = Structure(f"{PDB_ID}_target")
f_model = Model(0)
f_struct.add(f_model)

res_map = {}  # original target resnum -> new Chain B resnum (1..N)
target_resname_by_new = {}
ch_b = Chain("B")
for new_num, r in enumerate(target_aa_orig, start=1):
  res_map[r.id[1]] = new_num
  target_resname_by_new[new_num] = r.get_resname().strip().upper()
  ch_b.add(_copy_clean_residue(r, (" ", new_num, " ")))
f_model.add(ch_b)

chain_pool = list("CDEFGHIJKLMNOPQRSTUVWXYZ")
glycan_chains = {}   # chain_id -> [(new_resnum, resname), ...]
gidx_to_new = {}     # orig g_idx -> (new_chain_id, new_resnum, resname)

for tree in glycan_trees:
  g_cid = chain_pool.pop(0)
  ch_g = Chain(g_cid)
  glycan_chains[g_cid] = []
  for r_num, g_idx in enumerate(tree, start=1):
    g_res = all_glycans[g_idx][1]
    g_code = g_res.get_resname().strip().upper()
    ch_g.add(_copy_clean_residue(g_res, (f"H_{g_code}", r_num, " ")))
    glycan_chains[g_cid].append((r_num, g_code))
    gidx_to_new[g_idx] = (g_cid, r_num, g_code)
  f_model.add(ch_g)

ion_chain_id = None
if kept_ions:
  ion_chain_id = chain_pool.pop(0)
  ch_i = Chain(ion_chain_id)
  for idx, ion in enumerate(kept_ions, start=1):
    ch_i.add(_copy_clean_residue(ion, (f"H_{ion.get_resname().strip()}", idx, " ")))
  f_model.add(ch_i)

# Build covalent bond list & LINK records for the renumbered structure
covalent_bonds = []  # list of dicts for downstream AF3 / Boltz-2 / RF3
link_lines = []
seen_pairs = set()

for g_idx, (t_orig, a_t, a_g, d) in root_links.items():
  if g_idx in gidx_to_new and t_orig in res_map:
    b_num = res_map[t_orig]
    b_rname = target_resname_by_new[b_num]
    g_cid, g_num, g_rname = gidx_to_new[g_idx]
    covalent_bonds.append({
        "chain1": "B", "resnum1": b_num, "resname1": b_rname, "atom1": a_t,
        "chain2": g_cid, "resnum2": g_num, "resname2": g_rname, "atom2": a_g,
        "dist": round(d, 2),
    })
    link_lines.append(_format_link_line(a_t, b_rname, "B", b_num, a_g, g_rname, g_cid, g_num, d))

for i, neighbors in gly_adj.items():
  if i not in gidx_to_new:
    continue
  for j, a_i, a_j, d in neighbors:
    if j not in gidx_to_new or (min(i, j), max(i, j)) in seen_pairs:
      continue
    seen_pairs.add((min(i, j), max(i, j)))
    c1, n1, r1 = gidx_to_new[i]
    c2, n2, r2 = gidx_to_new[j]
    covalent_bonds.append({
        "chain1": c1, "resnum1": n1, "resname1": r1, "atom1": a_i,
        "chain2": c2, "resnum2": n2, "resname2": r2, "atom2": a_j,
        "dist": round(d, 2),
    })
    link_lines.append(_format_link_line(a_i, r1, c1, n1, a_j, r2, c2, n2, d))

target_ready_pdb = WORK_DIR / f"{PDB_ID}_target_ready.pdb"
io = PDBIO()
io.set_structure(f_struct)
io.save(str(target_ready_pdb))

# Prepend LINK records so RFD3/AtomWorks recognizes covalent glycan attachments
if link_lines:
  pdb_body = target_ready_pdb.read_text()
  target_ready_pdb.write_text("".join(link_lines) + pdb_body)

# Auto-build RFD3 target contig string (e.g., B1-118,C1-1,D1-1,E1-1,F1-1,G1-2)
contig_parts = [f"B1-{len(target_aa_orig)}"] + [
    f"{cid}1-{len(res_list)}" for cid, res_list in glycan_chains.items()
]
if ion_chain_id:
  contig_parts.append(f"{ion_chain_id}1-{len(kept_ions)}")
auto_target_contig = ",".join(contig_parts)

PREP_STATE = {
    "pdb_id": PDB_ID,
    "complex_pdb": complex_pdb,
    "target_ready_pdb": target_ready_pdb,
    "target_chain_orig": t_chain_id,
    "binder_chains_orig": b_chain_ids,
    "glycan_resnames": sorted(glycan_codes),
    "glycan_chains": glycan_chains,
    "covalent_bonds": covalent_bonds,
    "target_contig": auto_target_contig,
    "res_map": res_map,
}

print(f"✓ Saved RFD3-ready glycoprotein PDB: {target_ready_pdb}")
print(f"    Chain B (target): {len(target_aa_orig)} aa, renumbered B1–B{len(target_aa_orig)} (was {t_chain_id}{target_aa_orig[0].id[1]}–{t_chain_id}{target_aa_orig[-1].id[1]})")
for cid, res_list in glycan_chains.items():
  desc = "–".join(f"{rname}({rnum})" for rnum, rname in res_list)
  print(f"    Chain {cid} (glycan): {desc}")
if ion_chain_id:
  print(f"    Chain {ion_chain_id} (ions):   {', '.join(i.get_resname().strip() for i in kept_ions)}")
print(f"✓ Preserved {len(covalent_bonds)} covalent LINK record(s):")
for b in covalent_bonds:
  print(f"    {b['chain1']}{b['resnum1']}({b['resname1']}).{b['atom1']} — {b['chain2']}{b['resnum2']}({b['resname2']}).{b['atom2']} ({b['dist']} Å)")
print(f"✓ Suggested RFD3 target_contig: {auto_target_contig}")
if complex_pdb:
  print(f"✓ Saved intermediate complex PDB (for hotspot inspection): {complex_pdb}")

In [ ]:
#@title 3. Visualize Glycoprotein & Inspect Hotspots (py3Dmol)
view_mode = "Complex (with binder)" #@param ["Complex (with binder)", "Final Target (Chain B + Glycans)"] {type:"string"}
contact_cutoff = 5 #@param {type:"number"}
show_residue_labels = True #@param {type:"boolean"}
show_glycan_labels = True #@param {type:"boolean"}

import py3Dmol
from Bio.PDB import PDBParser, NeighborSearch

parser = PDBParser(QUIET=True)
glycan_codes = set(PREP_STATE["glycan_resnames"])
res_map = PREP_STATE["res_map"]

use_complex = (view_mode == "Complex (with binder)") and (PREP_STATE["complex_pdb"] is not None)
pdb_to_show = PREP_STATE["complex_pdb"] if use_complex else PREP_STATE["target_ready_pdb"]
model = parser.get_structure("vis", str(pdb_to_show))[0]

def _min_dist(res, other_atoms):
  return min(a - b for a in res.get_atoms() for b in other_atoms)

gly_residues = [
    (c.id, r) for c in model for r in c
    if r.id[0] != " " and r.get_resname().strip().upper() in glycan_codes
]
gly_atoms = [a for _, r in gly_residues for a in r.get_atoms()]

view = py3Dmol.view(width=800, height=550)
view.addModel(pdb_to_show.read_text(), "pdb")

if use_complex:
  t_cid = PREP_STATE["target_chain_orig"]
  b_cids = PREP_STATE["binder_chains_orig"]

  t_atoms = [a for r in model[t_cid] if r.id[0] == " " for a in r.get_atoms()]
  b_atoms = [a for bc in b_cids for r in model[bc] if r.id[0] == " " for a in r.get_atoms()]
  ns_t, ns_b = NeighborSearch(t_atoms), NeighborSearch(b_atoms)

  target_vs_b = sorted({r for a in b_atoms for r in ns_t.search(a.coord, contact_cutoff, level="R")}, key=lambda r: r.id[1])
  gly_vs_b = [
      (cid, r, _min_dist(r, b_atoms)) for cid, r in gly_residues
      if _min_dist(r, b_atoms) <= contact_cutoff
  ]

  print(f"=== Target ({t_cid} -> new Chain B) residues within {contact_cutoff} Å of Binder ({', '.join(b_cids)}) ===")
  rfd3_hotspots = []
  for r in target_vs_b:
    new_b = res_map[r.id[1]]
    rfd3_hotspots.append(f"B{new_b}")
    d_b = _min_dist(r, b_atoms)
    extra = f" | {_min_dist(r, gly_atoms):.2f} Å to glycan" if (gly_atoms and _min_dist(r, gly_atoms) <= contact_cutoff) else ""
    print(f"  B{new_b:<4} (was {t_cid}{r.id[1]:<4}) {r.get_resname():3s}  min dist to binder: {d_b:.2f} Å{extra}")
  print(f"\nRFD3 target hotspot candidates: {', '.join(rfd3_hotspots)}")

  if gly_vs_b:
    print(f"\n=== Glycan residues within {contact_cutoff} Å of Binder ({', '.join(b_cids)}) ===")
    for cid, r, d in gly_vs_b:
      print(f"  Chain {cid} res {r.id[1]} ({r.get_resname().strip()})  min dist to binder: {d:.2f} Å")

  # Style complex
  view.setStyle({"chain": t_cid}, {"cartoon": {"color": "#E8B4C8"}})
  for bc in b_cids:
    view.setStyle({"chain": bc}, {"cartoon": {"color": "#6BAED6", "opacity": 0.65}})

  if target_vs_b:
    view.addStyle({"chain": t_cid, "resi": [r.id[1] for r in target_vs_b]}, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.16}})
    if show_residue_labels:
      for r in target_vs_b:
        ca = r["CA"].coord
        view.addLabel(f"B{res_map[r.id[1]]} ({t_cid}{r.id[1]})", {"position": {"x": float(ca[0]), "y": float(ca[1]), "z": float(ca[2])}, "fontSize": 10, "fontColor": "#900C3F", "backgroundColor": "white", "backgroundOpacity": 0.7})

  for cid, r in gly_residues:
    view.setStyle({"chain": cid, "resi": r.id[1]}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.25}})
    if show_glycan_labels:
      c = sum(a.coord for a in r.get_atoms()) / len(list(r.get_atoms()))
      view.addLabel(f"{cid}{r.id[1]} {r.get_resname().strip()}", {"position": {"x": float(c[0]), "y": float(c[1]), "z": float(c[2])}, "fontSize": 10, "fontColor": "#0B6623", "backgroundColor": "white", "backgroundOpacity": 0.75})

  view.zoomTo({"chain": t_cid})

else:
  # Final target view (Chain B + all Glycan chains C, D, E...)
  view.setStyle({"chain": "B"}, {"cartoon": {"color": "#E8B4C8"}})

  # Highlight glycosylated ASN/SER/THR residues on Chain B
  glyc_sites = sorted({b["resnum1"] for b in PREP_STATE["covalent_bonds"] if b["chain1"] == "B"})
  if glyc_sites:
    view.addStyle({"chain": "B", "resi": glyc_sites}, {"stick": {"colorscheme": "magentaCarbon", "radius": 0.18}})
    if show_residue_labels:
      for rnum in glyc_sites:
        r = model["B"][(" ", rnum, " ")]
        ca = r["CA"].coord
        view.addLabel(f"B{rnum} {r.get_resname().strip()}", {"position": {"x": float(ca[0]), "y": float(ca[1]), "z": float(ca[2])}, "fontSize": 10, "fontColor": "#900C3F", "backgroundColor": "white", "backgroundOpacity": 0.75})

  for cid, r in gly_residues:
    view.setStyle({"chain": cid, "resi": r.id[1]}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.25}})
    if show_glycan_labels:
      atoms = list(r.get_atoms())
      c = sum(a.coord for a in atoms) / len(atoms)
      view.addLabel(f"{cid}{r.id[1]} {r.get_resname().strip()}", {"position": {"x": float(c[0]), "y": float(c[1]), "z": float(c[2])}, "fontSize": 10, "fontColor": "#0B6623", "backgroundColor": "white", "backgroundOpacity": 0.75})

  print(f"Displaying final glycoprotein target: {pdb_to_show}")
  print(f"  Target Chain B: {len(res_map)} aa | Glycosylation sites: {', '.join(f'B{n}' for n in glyc_sites)}")
  print(f"  Glycan chains:  {', '.join(f'{cid}1-{len(v)}' for cid, v in PREP_STATE['glycan_chains'].items())}")
  view.zoomTo({"chain": "B"})

view.show()

## Step 2: RFD3 Glycoprotein Binder Design — Recommended Parameters

Use the table below to configure the **RFD3** cell for the `8S6Z` (CD28 + `NAG`) example from the [Foundry Binder Design Tutorial](https://rosettacommons.github.io/foundry/models/rfd3/tutorials/binder_design_tutorial.html#binder-design-glycan-example):

| Parameter | Glycoprotein Exercise (`8S6Z` — CD28 + `NAG`) |
|---|---|
| **Job Name** | `cd28_nag_binder` |
| **Input PDB** | `glycan_binder_prep/8S6Z_target_ready.pdb` |
| **`binder_lengths`** | `55-88` |
| **`target_contig`** | `B1-118,C1-2,D1-1,E1-1,F1-1,G1` *(auto-detected from Cell 2)* |
| **`select_hotspots`** | `B3:CG1,CG2,CD1,CB; B4:CB,CD1,CD2,CG; B31:CB,CD,CG,CZ,NH1,NH2,NE; B100:CD1,CD2,CE1,CE2,CB,CZ,OH; B101:CB,CD,CG,N; B102:CB,CD,CG,N; B104:CD1,CD2,CE1,CE2,CB,CZ,OH; B105:CB,CD1,CD2,CG` |
| **`select_hbond_donor`** | `B31:NH1,NH2,NE; B100:OH; B104:OH` |
| **`select_hbond_acceptor`** | *(leave blank)* |
| **`infer_ori_strategy`** | `hotspots` |
| **`redesign_motif_sidechains`** | `False` |
| **`is_non_loopy`** | `True` |

### Quick Notes on Glycoprotein Conditioning
- **Explicit Glycan Chains in `contig`:** Listing every glycan chain (`C1-2,D1-1,E1-1,F1-1,G1`) ensures RFD3 sees the full glycan shield during diffusion and designs binders that pack cleanly around it without clashing.
- **Hotspot Residues:** All hotspots above target the CD28 protein surface (Chain `B`), using the Fab-binding epitope while keeping the neighboring `NAG` chains fixed in place.
- **MPNN Checkpoint:** Because non-protein `NAG` atoms are present in the RFD3 output `.cif.gz`, use **LigandMPNN** in the next step so sequence design accounts for the glycan atoms.

In [ ]:
#@title Run RFDiffusion3 — Glycoprotein Binder Design
#@markdown Designs *de novo* binders against a glycosylated protein.

#@markdown ### Execution Mode
execution_mode = "Run in Colab" #@param ["Run in Colab", "Prepare inputs & print terminal commands"] {type:"string"}

#@markdown ### Input & Output
job_name = "cd28_nag_binder" #@param {type:"string"}
#@markdown Leave `input_pdb` blank to automatically use `PREP_STATE["target_ready_pdb"]` from Cell 2.
input_pdb = "" #@param {type:"string"}
out_dir = "8S6Z_binder_design/rfd3_out" #@param {type:"string"}

#@markdown ### Binder Size & Contig
#@markdown Binder length: single (`"70"`), range (`"55-88"`), or comma-separated (`"55-88, 90-110"`).
binder_lengths = "55-88" #@param {type:"string"}
#@markdown Leave `target_contig` blank to auto-detect from `input_pdb` (e.g. `B1-118,C1-2,D1-1,E1-1,F1-1,G1-1`), or enter manually.
target_contig = "" #@param {type:"string"}
#@markdown Total designs per size entry = `n_batches × diffusion_batch_size`.
n_batches = 10 #@param {type:"integer"}
diffusion_batch_size = 1 #@param {type:"integer"}

#@markdown ### Binder & Hydrogen-Bond Conditioning
redesign_motif_sidechains = False #@param {type:"boolean"}
infer_ori_strategy = "hotspots" #@param ["hotspots", "com", "none"] {type:"string"}
#@markdown **Hotspots** — use `RES:ATOM1,ATOM2; RES2:ATOM3` (or residue-only `B64, B87, C1`, or a JSON dict). Leave blank to omit.
select_hotspots = "B3:CG1,CG2,CD1,CB; B4:CB,CD1,CD2,CG; B31:CB,CD,CG,CZ,NH1,NH2,NE; B100:CD1,CD2,CE1,CE2,CB,CZ,OH; B101:CB,CD,CG,N; B102:CB,CD,CG,N; B104:CD1,CD2,CE1,CE2,CB,CZ,OH; B105:CB,CD1,CD2,CG" #@param {type:"string"}
#@markdown **H-bond donor conditioning** (atoms on target/ligand that should donate an H-bond to the binder):
select_hbond_donor = "B31:NH1,NH2,NE; B100:OH; B104:OH" #@param {type:"string"}
#@markdown **H-bond acceptor conditioning** (optional, same format; leave blank to omit):
select_hbond_acceptor = "" #@param {type:"string"}

#@markdown ### Designability vs. Diversity (Sampling)
#@markdown - **Balanced (default):** `step_scale=1.5`, `gamma_0=0.6`, `is_non_loopy=True`
#@markdown - **High designability (low-T):** `step_scale=3.0`, `gamma_0=0.2`, `is_non_loopy=True`
#@markdown - **High diversity:** `step_scale=1.0`, `gamma_0=0.8`, `is_non_loopy=False`
step_scale = 1.5 #@param {type:"number"}
gamma_0 = 0.6 #@param {type:"number"}
num_timesteps = 200 #@param {type:"integer"}
is_non_loopy = True #@param {type:"boolean"}

#@markdown ### Execution & Memory
low_memory_mode = False #@param {type:"boolean"}
dump_trajectories = False #@param {type:"boolean"}
skip_existing = False #@param {type:"boolean"}

import json, os, pathlib, shlex
from Bio.PDB import PDBParser, MMCIFParser

# 1. Resolve target PDB path
if input_pdb.strip():
  target_path = pathlib.Path(input_pdb.strip()).resolve()
elif "PREP_STATE" in globals() and PREP_STATE.get("target_ready_pdb"):
  target_path = pathlib.Path(PREP_STATE["target_ready_pdb"]).resolve()
else:
  raise ValueError("No input PDB found. Run Cell 2 first or enter a path in input_pdb.")

if not target_path.exists():
  raise FileNotFoundError(f"Input PDB not found: {target_path}")

# 2. Auto-detect target contig (e.g., B1-167,C1-1) if left blank
if target_contig.strip():
  tgt_contig = target_contig.strip()
else:
  parser = MMCIFParser(QUIET=True) if target_path.suffix.lower() == ".cif" else PDBParser(QUIET=True)
  model = parser.get_structure("tgt", str(target_path))[0]
  parts = []
  for chain in model:
    res_nums = [r.id[1] for r in chain if r.id[0] != "W" and r.get_resname().strip() != "HOH"]
    if res_nums:
      parts.append(f"{chain.id}{min(res_nums)}-{max(res_nums)}")
  tgt_contig = ",".join(parts)

# 3. Parse atom/residue selections (supports JSON dict, 'B64:OE1,OE2; C1:O20', or 'B64, B87')
def _parse_selection(text: str):
  s = text.strip()
  if not s:
    return None
  if s.startswith("{"):
    return json.loads(s)
  if ":" not in s:
    return ",".join(tok.strip() for tok in s.replace(";", ",").split(",") if tok.strip())
  sel = {}
  for item in s.split(";"):
    item = item.strip()
    if not item:
      continue
    if ":" in item:
      res_id, atoms = item.split(":", 1)
      sel[res_id.strip()] = ",".join(a.strip() for a in atoms.split(",") if a.strip())
    else:
      sel[item] = "ALL"
  return sel

hotspots = _parse_selection(select_hotspots)
hbond_donor = _parse_selection(select_hbond_donor)
hbond_acceptor = _parse_selection(select_hbond_acceptor)

# 4. Build RFD3 JSON specification(s)
length_entries = [x.strip() for x in binder_lengths.split(",") if x.strip()]
specs = {}
for idx, entry in enumerate(length_entries, start=1):
  key = job_name if len(length_entries) == 1 else f"{job_name}_{idx}"
  spec = {
      "input": str(target_path),
      "contig": f"{entry},/0,{tgt_contig}",
      "redesign_motif_sidechains": redesign_motif_sidechains,
      "is_non_loopy": is_non_loopy,
  }
  if infer_ori_strategy != "none":
    spec["infer_ori_strategy"] = infer_ori_strategy
  if hotspots is not None:
    spec["select_hotspots"] = hotspots
  if hbond_donor is not None:
    spec["select_hbond_donor"] = hbond_donor
  if hbond_acceptor is not None:
    spec["select_hbond_acceptor"] = hbond_acceptor
  specs[key] = spec

rfd3_out = pathlib.Path(out_dir).resolve()
rfd3_out.mkdir(parents=True, exist_ok=True)

config_path = rfd3_out.parent / f"{job_name}.json"
config_path.write_text(json.dumps(specs, indent=2) + "\n")

total_designs = len(specs) * n_batches * diffusion_batch_size
print(f"Config saved to {config_path} ({len(specs)} spec(s) × {n_batches} batch(es) × {diffusion_batch_size} = {total_designs} designs)")
print(json.dumps(specs, indent=2))

# 5. Build RFD3 command and either print terminal instructions or run
cmd = [
    "rfd3", "design",
    f"inputs={config_path.resolve()}",
    f"out_dir={rfd3_out}",
    'global_prefix=""',
    f"n_batches={n_batches}",
    f"diffusion_batch_size={diffusion_batch_size}",
    f"inference_sampler.num_timesteps={num_timesteps}",
    f"inference_sampler.step_scale={step_scale}",
    f"inference_sampler.gamma_0={gamma_0}",
    f"low_memory_mode={low_memory_mode}",
    f"dump_trajectories={dump_trajectories}",
    f"skip_existing={skip_existing}",
    "prevalidate_inputs=True",
]

if execution_mode == "Prepare inputs & print terminal commands":
  env_path = pathlib.Path(os.environ["ENVS_DIR"]) / "foundry"
  cmd_str = " \\\n  ".join(shlex.quote(c) for c in cmd)
  script = f"""#!/usr/bin/env bash
set -euo pipefail

# Activate Foundry environment & set required paths
export VIRTUAL_ENV={shlex.quote(str(env_path))}
export PATH="{env_path}/bin:$PATH"
export FOUNDRY_CHECKPOINT_DIRS={shlex.quote(os.environ['FOUNDRY_CHECKPOINT_DIRS'])}
export CCD_MIRROR_PATH=/tmp
export PDB_MIRROR_PATH=/tmp
export PYTHONWARNINGS=ignore

# Run RFdiffusion3
{cmd_str}
"""
  sh_path = rfd3_out.parent / f"run_rfd3_{job_name}.sh"
  sh_path.write_text(script)
  sh_path.chmod(0o755)
  print("\n" + "=" * 72)
  print("TERMINAL INSTRUCTIONS — Copy & paste into a terminal, or run:")
  print(f"  bash {sh_path}")
  print("=" * 72)
  print(script)
else:
  run_command_with_env("foundry", cmd)

In [ ]:
#@title Check & Filter RFD3 Interface / Conditioned H-Bonds (HBPLUS)
#@markdown Runs **HBPLUS** on all RFD3 designs against the glycoprotein before MPNN, and optionally exports the top-*k* ranked designs to a separate folder for MPNN/refolding.

#@markdown ### Input & Conditioning Options
#@markdown Leave `rfd3_dir` blank to automatically use `rfd3_out` / `out_dir` from the RFD3 cell.
rfd3_dir = "8S6Z_binder_design/rfd3_out" #@param {type:"string"}
binder_chain = "A" #@param {type:"string"}
#@markdown Optional overrides (leave blank to auto-read `select_hbond_donor` / `select_hbond_acceptor` from each design's `.json` metadata):
override_hbond_donor = "" #@param {type:"string"}
override_hbond_acceptor = "" #@param {type:"string"}
#@markdown If `False` (recommended), counts any interface H-bond involving a conditioned target atom regardless of HBPLUS protonation assignment. If `True`, requires exact donor/acceptor direction.
strict_donor_acceptor_role = False #@param {type:"boolean"}

#@markdown ### Filtering, Top-*K* Export & Display
min_interface_hbonds = 0 #@param {type:"integer"}
min_conditioned_hits = 0 #@param {type:"integer"}
#@markdown Keep only the top-*k* ranked passing designs (`0` = keep all passing designs). Useful for capping `8S6Z` refolding at 5 designs.
top_k = 0 #@param {type:"integer"}
#@markdown Copy selected designs (`.cif.gz`/`.pdb` + companion `.json`) into `<rfd3_dir>_selected` for MPNN/refolding:
export_selected = True #@param {type:"boolean"}
show_all_interface_hbonds = True #@param {type:"boolean"}

import gzip, json, os, pathlib, re, shutil, subprocess, tempfile
import numpy as np
import pandas as pd
from Bio.PDB import MMCIFParser, PDBParser, PDBIO

# 1. Resolve RFD3 output directory & HBPLUS binary
if rfd3_dir.strip():
  out_path = pathlib.Path(rfd3_dir.strip()).resolve()
elif "rfd3_out" in globals() and pathlib.Path(rfd3_out).exists():
  out_path = pathlib.Path(rfd3_out).resolve()
elif "out_dir" in globals() and pathlib.Path(out_dir).exists():
  out_path = pathlib.Path(out_dir).resolve()
else:
  raise ValueError("Could not locate RFD3 output directory. Run the RFD3 cell first or set rfd3_dir.")

hbplus_bin = os.environ.get("HBPLUS_PATH", "/usr/local/bin/hbplus")
if not (hbplus_bin and os.access(hbplus_bin, os.X_OK)):
  raise FileNotFoundError(f"HBPLUS executable not found or not executable at: {hbplus_bin}")

struct_files = sorted(
    list(out_path.glob("*.cif.gz")) + list(out_path.glob("*.cif")) + list(out_path.glob("*.pdb"))
)
if not struct_files:
  raise FileNotFoundError(f"No .cif.gz, .cif, or .pdb files found in {out_path}")


def _parse_override(text: str):
  s = (text or "").strip()
  if not s:
    return None
  if s.startswith("{"):
    return json.loads(s)
  if ":" not in s:
    return ",".join(tok.strip() for tok in s.replace(";", ",").split(",") if tok.strip())
  sel = {}
  for item in s.split(";"):
    item = item.strip()
    if not item:
      continue
    if ":" in item:
      res_id, atoms = item.split(":", 1)
      sel[res_id.strip()] = ",".join(a.strip() for a in atoms.split(",") if a.strip())
    else:
      sel[item] = "ALL"
  return sel


def _normalize_cond_spec(spec):
  out = {}
  if not spec:
    return out
  items = {tok.strip(): "ALL" for tok in spec.replace(";", ",").split(",") if tok.strip()} if isinstance(spec, str) else spec
  if not isinstance(items, dict):
    return out
  for res_key, atoms in items.items():
    m = re.match(r"^([A-Za-z0-9])(\d+)$", str(res_key).strip())
    if not m:
      continue
    key = (m.group(1), int(m.group(2)))
    if atoms is None or str(atoms).strip().upper() in ("", "ALL", "*"):
      out[key] = None
    else:
      atom_list = atoms if isinstance(atoms, (list, tuple, set)) else str(atoms).split(",")
      atom_set = {str(a).strip().upper() for a in atom_list if str(a).strip()}
      out[key] = (out[key] | atom_set) if (key in out and out[key] is not None) else atom_set
  return out


def _matches_cond(cond_map, ch, resnum, atom):
  if (ch, resnum) not in cond_map:
    return False
  allowed = cond_map[(ch, resnum)]
  return (allowed is None) or (atom.upper() in allowed)


def _load_structure(path: pathlib.Path):
  name = path.name.lower()
  if name.endswith(".cif.gz"):
    with gzip.open(path, "rt") as fh:
      struct = MMCIFParser(QUIET=True).get_structure("des", fh)
  elif name.endswith(".cif"):
    struct = MMCIFParser(QUIET=True).get_structure("des", str(path))
  else:
    struct = PDBParser(QUIET=True).get_structure("des", str(path))

  for model in struct:
    for chain in model:
      for res in list(chain):
        for atom in list(res):
          if not np.all(np.isfinite(atom.get_coord())):
            res.detach_child(atom.get_id())
        if len(res) == 0:
          chain.detach_child(res.get_id())
  return struct


def _meta_path_for(path: pathlib.Path) -> pathlib.Path:
  stem = path.name[:-7] if path.name.lower().endswith(".cif.gz") else path.stem
  return path.with_name(f"{stem}.json")


def _run_hbplus(struct, hbplus_exe: str):
  with tempfile.TemporaryDirectory() as tmp:
    pdb_path = pathlib.Path(tmp) / "design.pdb"
    io_pdb = PDBIO()
    io_pdb.set_structure(struct)
    io_pdb.save(str(pdb_path))
    subprocess.run([hbplus_exe, str(pdb_path)], cwd=tmp, capture_output=True, text=True, check=True)
    hb2_path = pathlib.Path(tmp) / "design.hb2"
    raw_lines = hb2_path.read_text().splitlines()[8:] if hb2_path.exists() else []

  pairs = {}
  for line in raw_lines:
    if len(line) < 35:
      continue
    try:
      d_node = (line[0].strip(), int(line[1:5]), line[6:9].strip(), line[10:14].strip())
      a_node = (line[14].strip(), int(line[15:19]), line[20:23].strip(), line[24:28].strip())
      dist, cat = float(line[28:33].strip()), line[33:35].strip()
    except ValueError:
      continue
    key = frozenset([(d_node[0], d_node[1], d_node[3]), (a_node[0], a_node[1], a_node[3])])
    if key not in pairs:
      pairs[key] = {"donor": d_node, "acceptor": a_node, "dist": dist, "cat": cat, "bidirectional": False}
    else:
      pairs[key]["bidirectional"] = True
      pairs[key]["dist"] = min(pairs[key]["dist"], dist)
  return list(pairs.values())


# 2. Screen all designs
ov_donor = _parse_override(override_hbond_donor)
ov_acceptor = _parse_override(override_hbond_acceptor)
rows = []

for spath in struct_files:
  struct = _load_structure(spath)
  model = struct[0]
  design_name = spath.name[:-7] if spath.name.lower().endswith(".cif.gz") else spath.stem

  is_het, binder_len = {}, 0
  for ch in model:
    for res in ch:
      if res.id[0] == "W" or res.get_resname().strip() == "HOH":
        continue
      is_het[(ch.id, res.id[1])] = (res.id[0] != " ")
      if ch.id == binder_chain:
        binder_len += 1

  meta_path = _meta_path_for(spath)
  spec = json.loads(meta_path.read_text()).get("specification", {}) if meta_path.exists() else {}
  cond_donors = _normalize_cond_spec(ov_donor if ov_donor is not None else spec.get("select_hbond_donor"))
  cond_acceptors = _normalize_cond_spec(ov_acceptor if ov_acceptor is not None else spec.get("select_hbond_acceptor"))

  inter_details, cond_details = [], []
  n_prot, n_lig = 0, 0

  for hb in _run_hbplus(struct, hbplus_bin):
    d_ch, a_ch = hb["donor"][0], hb["acceptor"][0]
    if (d_ch == binder_chain) == (a_ch == binder_chain):
      continue

    if d_ch == binder_chain:
      b_node, t_node = hb["donor"], hb["acceptor"]
      t_role = "both" if hb["bidirectional"] else "acceptor"
      arrow = "<->" if hb["bidirectional"] else "<-"
    else:
      t_node, b_node = hb["donor"], hb["acceptor"]
      t_role = "both" if hb["bidirectional"] else "donor"
      arrow = "<->" if hb["bidirectional"] else "->"

    t_ch, t_res, t_resn, t_atom = t_node
    b_ch, b_res, b_resn, b_atom = b_node
    if is_het.get((t_ch, t_res), False):
      n_lig += 1
    else:
      n_prot += 1

    label = f"{t_ch}{t_res}({t_resn}):{t_atom} {arrow} {b_ch}{b_res}({b_resn}):{b_atom} ({hb['dist']:.2f}Å, {hb['cat']})"
    inter_details.append(label)

    hit_donor = _matches_cond(cond_donors, t_ch, t_res, t_atom) and (not strict_donor_acceptor_role or t_role in ("donor", "both"))
    hit_acceptor = _matches_cond(cond_acceptors, t_ch, t_res, t_atom) and (not strict_donor_acceptor_role or t_role in ("acceptor", "both"))
    if hit_donor or hit_acceptor:
      cond_details.append(label)

  n_inter = n_prot + n_lig
  n_cond = len(cond_details)
  rows.append({
      "design": design_name,
      "binder_len": binder_len,
      "interface_hbonds": n_inter,
      "protein_hbonds": n_prot,
      "ligand_hbonds": n_lig,
      "conditioned_hits": n_cond,
      "pass_filter": (n_inter >= min_interface_hbonds) and (n_cond >= min_conditioned_hits),
      "conditioned_details": "; ".join(cond_details),
      "all_interface_details": "; ".join(inter_details),
      "cif_path": str(spath),
  })

# 3. Rank, select top-k, save CSV, and optionally export
hbplus_df = pd.DataFrame(rows).sort_values(
    by=["pass_filter", "conditioned_hits", "ligand_hbonds", "interface_hbonds", "design"],
    ascending=[False, False, False, False, True],
).reset_index(drop=True)

passing_idx = hbplus_df.index[hbplus_df["pass_filter"]].tolist()
selected_idx = passing_idx[:top_k] if top_k > 0 else passing_idx
hbplus_df["selected"] = hbplus_df.index.isin(selected_idx)

csv_path = out_path.parent / "rfd3_hbplus_summary.csv"
hbplus_df.to_csv(csv_path, index=False)

print(f"Analyzed {len(hbplus_df)} designs in {out_path} -> saved summary to {csv_path}")
print(f"Passing thresholds: {len(passing_idx)} | Selected for MPNN (top_k={top_k or 'all'}): {len(selected_idx)}\n")

for _, r in hbplus_df.iterrows():
  flag = "SEL " if r["selected"] else ("PASS" if r["pass_filter"] else "FAIL")
  print(
      f"[{flag}] {r['design']:28s} | len={r['binder_len']:3d} | "
      f"interface={r['interface_hbonds']:2d} (prot={r['protein_hbonds']}, lig={r['ligand_hbonds']}) | "
      f"conditioned={r['conditioned_hits']}"
  )
  if r["conditioned_details"]:
    for item in r["conditioned_details"].split("; "):
      print(f"    * [COND] {item}")
  elif show_all_interface_hbonds and r["all_interface_details"]:
    for item in r["all_interface_details"].split("; "):
      print(f"      [INTF] {item}")

selected_dir = out_path.parent / f"{out_path.name}_selected"
if export_selected:
  if selected_dir.exists():
    shutil.rmtree(selected_dir)
  selected_dir.mkdir(parents=True, exist_ok=True)
  for sp in hbplus_df.loc[hbplus_df["selected"], "cif_path"]:
    src = pathlib.Path(sp)
    shutil.copy2(src, selected_dir / src.name)
    mpath = _meta_path_for(src)
    if mpath.exists():
      shutil.copy2(mpath, selected_dir / mpath.name)
  print(f"\nExported {int(hbplus_df['selected'].sum())} selected design(s) + JSON metadata to:\n  {selected_dir}")

HBPLUS_RFD3_STATE = {
    "csv_path": str(csv_path),
    "selected_dir": str(selected_dir) if export_selected else str(out_path),
    "df": hbplus_df,
    "selected_cifs": hbplus_df.loc[hbplus_df["selected"], "cif_path"].tolist(),
}

In [ ]:
#@title Run MPNN — Binder Inverse Folding (via Foundry)
#@markdown Redesigns amino acid sequences directly on RFD3 `.cif` / `.cif.gz` backbones while keeping the glycosylated target protein fixed.

#@markdown ### Execution Mode
execution_mode = "Run in Colab" #@param ["Run in Colab", "Prepare inputs & print terminal commands"] {type:"string"}

#@markdown ### Input & Output
#@markdown Leave `input_dir` blank to automatically use `rfd3_out` from the RFD3 cell above.
input_dir = "8S6Z_binder_design/rfd3_out" #@param {type:"string"}
out_dir = "8S6Z_binder_design/mpnn_out" #@param {type:"string"}

#@markdown ### Model & Sampling
#@markdown Use **ProteinMPNN** or **SolubleMPNN** for protein-only targets, and **LigandMPNN** whenever a small molecule, ion, or nucleic acid is present.
mpnn_model = "SolubleMPNN (solublempnn_v_48_020.pt)" #@param ["SolubleMPNN (solublempnn_v_48_002.pt)", "SolubleMPNN (solublempnn_v_48_010.pt)", "SolubleMPNN (solublempnn_v_48_020.pt)", "SolubleMPNN (solublempnn_v_48_030.pt)", "LigandMPNN (ligandmpnn_v_32_005_25.pt)", "LigandMPNN (ligandmpnn_v_32_010_25.pt)", "LigandMPNN (ligandmpnn_v_32_020_25.pt)", "LigandMPNN (ligandmpnn_v_32_030_25.pt)", "ProteinMPNN (proteinmpnn_v_48_002.pt)", "ProteinMPNN (proteinmpnn_v_48_010.pt)", "ProteinMPNN (proteinmpnn_v_48_020.pt)", "ProteinMPNN (proteinmpnn_v_48_030.pt)"] {type:"string"}
#@markdown Total sequences per backbone = `batch_size × number_of_batches`
batch_size = 2 #@param {type:"integer"}
number_of_batches = 1 #@param {type:"integer"}
temperature = 0.1 #@param {type:"number"}
seed = 0 #@param {type:"integer"}

#@markdown ### Chain / Residue Selection
#@markdown Foundry MPNN accepts **one** selection mode at a time. For binder design, use `designed_chains` = `A` (fixes target Chain B & glycan chains C-G).
selection_mode = "designed_chains" #@param ["designed_chains", "fixed_chains", "designed_residues", "fixed_residues", "all (unconditional)"] {type:"string"}
#@markdown Comma-separated chains (e.g., `A`) or residues (e.g., `A1, A2, B42`). Ignored if `all (unconditional)`.
selection_value = "A" #@param {type:"string"}
#@markdown Comma-separated 3-letter residues to omit (e.g., `CYS` to avoid unwanted disulfides), or leave blank.
omit_residues = "CYS" #@param {type:"string"}

import json, os, pathlib, shlex

MPNN_MODELS = {
    "SolubleMPNN (solublempnn_v_48_002.pt)": ("solublempnn_v_48_002.pt", "protein_mpnn"),
    "SolubleMPNN (solublempnn_v_48_010.pt)": ("solublempnn_v_48_010.pt", "protein_mpnn"),
    "SolubleMPNN (solublempnn_v_48_020.pt)": ("solublempnn_v_48_020.pt", "protein_mpnn"),
    "SolubleMPNN (solublempnn_v_48_030.pt)": ("solublempnn_v_48_030.pt", "protein_mpnn"),
    "LigandMPNN (ligandmpnn_v_32_005_25.pt)": ("ligandmpnn_v_32_005_25.pt", "ligand_mpnn"),
    "LigandMPNN (ligandmpnn_v_32_010_25.pt)": ("ligandmpnn_v_32_010_25.pt", "ligand_mpnn"),
    "LigandMPNN (ligandmpnn_v_32_020_25.pt)": ("ligandmpnn_v_32_020_25.pt", "ligand_mpnn"),
    "LigandMPNN (ligandmpnn_v_32_030_25.pt)": ("ligandmpnn_v_32_030_25.pt", "ligand_mpnn"),
    "ProteinMPNN (proteinmpnn_v_48_002.pt)": ("proteinmpnn_v_48_002.pt", "protein_mpnn"),
    "ProteinMPNN (proteinmpnn_v_48_010.pt)": ("proteinmpnn_v_48_010.pt", "protein_mpnn"),
    "ProteinMPNN (proteinmpnn_v_48_020.pt)": ("proteinmpnn_v_48_020.pt", "protein_mpnn"),
    "ProteinMPNN (proteinmpnn_v_48_030.pt)": ("proteinmpnn_v_48_030.pt", "protein_mpnn"),
}

ckpt_name, model_type = MPNN_MODELS[mpnn_model]
ckpt_path = pathlib.Path(os.environ["FOUNDRY_CHECKPOINT_DIRS"]) / ckpt_name
if not ckpt_path.exists():
  raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}. Re-run the Setup cell.")

resolved_input = input_dir.strip() or (str(rfd3_out) if "rfd3_out" in globals() else "")
if not resolved_input:
  raise ValueError("No input directory specified. Run the RFD3 cell first or enter a path in input_dir.")

rfd3_dir = pathlib.Path(resolved_input).resolve()
cif_files = sorted(rfd3_dir.glob("*.cif*")) if rfd3_dir.exists() else []

ligandmpnn_out = pathlib.Path(out_dir).resolve()
ligandmpnn_out.mkdir(parents=True, exist_ok=True)

sel_items = [x.strip() for x in selection_value.split(",") if x.strip()]
omit_list = [r.strip().upper() for r in omit_residues.split(",") if r.strip()]

if execution_mode == "Prepare inputs & print terminal commands":
  env_path = pathlib.Path(os.environ["ENVS_DIR"]) / "foundry"
  extra_args = [
      f"--out_directory {shlex.quote(str(ligandmpnn_out))}",
      f"--checkpoint_path {shlex.quote(str(ckpt_path))}",
      f"--model_type {shlex.quote(model_type)}",
      "--is_legacy_weights True",
      f"--batch_size {int(batch_size)}",
      f"--number_of_batches {int(number_of_batches)}",
      f"--temperature {float(temperature)}",
      f"--seed {int(seed)}",
  ]
  if selection_mode != "all (unconditional)" and sel_items:
    val = sel_items[0] if len(sel_items) == 1 else json.dumps(sel_items)
    extra_args.append(f"--{selection_mode} {shlex.quote(val)}")
  if omit_list:
    extra_args.append(f"--omit {shlex.quote(json.dumps(omit_list))}")
  args_block = " \\\n    ".join(extra_args)

  script = f"""#!/usr/bin/env bash
set -euo pipefail

# Activate Foundry environment & set required paths
export VIRTUAL_ENV={shlex.quote(str(env_path))}
export PATH="{env_path}/bin:$PATH"
export FOUNDRY_CHECKPOINT_DIRS={shlex.quote(os.environ['FOUNDRY_CHECKPOINT_DIRS'])}
export CCD_MIRROR_PATH=/tmp
export PDB_MIRROR_PATH=/tmp
export PYTHONWARNINGS=ignore

# Run MPNN over all RFD3 .cif / .cif.gz backbones
mkdir -p {shlex.quote(str(ligandmpnn_out))}
shopt -s nullglob
for cif in {shlex.quote(str(rfd3_dir))}/*.cif*; do
  echo "Designing sequences for: $cif"
  mpnn \\
    --structure_path "$cif" \\
    {args_block}
done
"""
  sh_path = ligandmpnn_out.parent / "run_mpnn.sh"
  sh_path.write_text(script)
  sh_path.chmod(0o755)
  print(f"Found {len(cif_files)} backbone(s) currently in {rfd3_dir} (loop will process all *.cif* present when run).")
  print("\n" + "=" * 72)
  print("TERMINAL INSTRUCTIONS — Copy & paste into a terminal, or run:")
  print(f"  bash {sh_path}")
  print("=" * 72)
  print(script)
else:
  if not cif_files:
    raise FileNotFoundError(f"No .cif or .cif.gz files found in {rfd3_dir}")
  print(f"Running {mpnn_model} on {len(cif_files)} backbone(s) -> {ligandmpnn_out}", flush=True)
  if selection_mode != "all (unconditional)" and sel_items:
    print(f"Selection: --{selection_mode} {sel_items}", flush=True)

  for cif_path in cif_files:
    cmd = [
        "mpnn",
        "--structure_path", str(cif_path.resolve()),
        "--out_directory", str(ligandmpnn_out),
        "--checkpoint_path", str(ckpt_path),
        "--model_type", model_type,
        "--is_legacy_weights", "True",
        "--batch_size", str(int(batch_size)),
        "--number_of_batches", str(int(number_of_batches)),
        "--temperature", str(float(temperature)),
        "--seed", str(int(seed)),
    ]
    if selection_mode != "all (unconditional)" and sel_items:
      val = sel_items[0] if len(sel_items) == 1 else json.dumps(sel_items)
      cmd += [f"--{selection_mode}", val]
    if omit_list:
      cmd += ["--omit", json.dumps(omit_list)]

    run_command_with_env("foundry", cmd)

In [ ]:
#@title Run Refolding — Glycoprotein Complex Prediction (AlphaFold3 / OpenFold3 / Boltz-2 / RosettaFold3)
#@markdown Refolds binder + glycosylated target complexes (preserving all `ASN–glycan` and `glycan–glycan` covalent bonds) from MPNN FASTA outputs.

#@markdown ### Execution Mode
execution_mode = "Run in Colab" #@param ["Run in Colab", "Prepare inputs & print terminal commands"] {type:"string"}

#@markdown ### Model & Paths
#@markdown Leave as `Auto (from Setup)` to use the model installed in the Setup cell.
model_choice = "Auto (from Setup)" #@param ["Auto (from Setup)", "AlphaFold3", "OpenFold3", "Boltz2", "RosettaFold3"] {type:"string"}
#@markdown Leave `mpnn_dir` blank to automatically use `mpnn_out` from the MPNN cell above.
mpnn_dir = "" #@param {type:"string"}
#@markdown Leave `out_dir` blank to auto-name (e.g., `8S6Z_binder_design/<model>_out`).
out_dir = "" #@param {type:"string"}
#@markdown Which designed sequence to refold per backbone (`1` = first designed sequence; skips entry 0 if native/poly-Gly).
seq_index = 1 #@param {type:"integer"}
on_existing = "overwrite" #@param ["overwrite", "skip"] {type:"string"}

#@markdown ### Glycans, Additional Ligands & MSA
#@markdown Automatically include all glycan chains (`C..G`) and covalent bonds detected in Cell 2 (`PREP_STATE`).
include_glycans = True #@param {type:"boolean"}
#@markdown Optional: extra non-covalent colon-separated **CCD codes** (e.g., `ZN` or `BB2`), or leave blank.
ligand_ccd = "" #@param {type:"string"}
#@markdown Optional: extra colon-separated **SMILES strings** (case-sensitive; for custom ligands not in PDB CCD).
ligand_smiles = "" #@param {type:"string"}
#@markdown If `True`, converts extra `ligand_ccd` entries to SMILES via RCSB (glycans always remain CCD).
use_smiles_from_ccd = False #@param {type:"boolean"}
#@markdown - `binder_single_target_msa`: single sequence for Chain A (binder), 1 cached MMseqs2 MSA call for shared Chain B+ (target).
#@markdown - `mmseqs2_server`: cached MMseqs2 MSA for all unique protein chains.
#@markdown - `single_sequence`: skip MSA for all chains (fastest).
msa_mode = "binder_single_target_msa" #@param ["binder_single_target_msa", "mmseqs2_server", "single_sequence"] {type:"string"}

#@markdown ### Shared Sampling Settings
num_recycles = 10 #@param {type:"integer"}
num_diffusion_samples = 5 #@param {type:"integer"}
seeds = "1" #@param {type:"string"}

#@markdown ### Boltz-2 Specific Options *(ignored for AF3 / OpenFold3 / RF3)*
boltz_sampling_steps = 200 #@param {type:"integer"}
boltz_step_scale = 1.5 #@param {type:"number"}
boltz_use_potentials = False #@param {type:"boolean"}
boltz_output_format = "mmcif" #@param ["mmcif", "pdb"] {type:"string"}

#@markdown ### RosettaFold3 Specific Options *(ignored for AF3 / OpenFold3 / Boltz-2)*
#@markdown Diffusion steps (`200` is standard; `50` is >2x faster with similar accuracy).
rf3_num_steps = 200 #@param {type:"integer"}
#@markdown Set to `0.0` to always output structures for single-sequence refolding (avoids RF3 early stopping).
rf3_early_stopping_plddt = 0.0 #@param {type:"number"}

import copy, io, json, os, pathlib, re, shlex, shutil, subprocess, tarfile, time, urllib.parse, urllib.request

# 1. Resolve selected model & paths
selected = os.environ.get("FOLDING_MODEL", "AlphaFold3") if model_choice == "Auto (from Setup)" else model_choice
model_key = {
    "AlphaFold3": "alphafold3",
    "OpenFold3": "openfold3",
    "Boltz2": "boltz2",
    "RosettaFold3": "rosettafold3",
}[selected]
env_name = {"Boltz2": "boltz", "RosettaFold3": "foundry"}.get(selected, "af3")

if not (pathlib.Path(os.environ["ENVS_DIR"]) / env_name).exists():
  raise RuntimeError(f"Environment '{env_name}' is not installed. Select '{selected}' in the Setup cell and re-run it.")

if selected == "RosettaFold3":
  ckpt_dir = pathlib.Path(os.environ.get("FOUNDRY_CHECKPOINT_DIRS", "/content/foundry/checkpoints"))
  rf3_ckpts = list(ckpt_dir.glob("rf3*.pt")) + list(ckpt_dir.glob("rf3*.ckpt"))
  if not rf3_ckpts:
    raise RuntimeError(f"No RF3 checkpoint found in {ckpt_dir}. Select 'RosettaFold3' in the Setup cell and re-run it.")

resolved_mpnn = mpnn_dir.strip() or (str(ligandmpnn_out) if "ligandmpnn_out" in globals() else "")
if not resolved_mpnn:
  raise ValueError("No MPNN directory specified. Run the MPNN cell first or enter a path in mpnn_dir.")
mpnn_path = pathlib.Path(resolved_mpnn).resolve()

pred_out = pathlib.Path(out_dir.strip()).resolve() if out_dir.strip() else (mpnn_path.parent / f"{model_key}_out").resolve()
inputs_dir = pred_out / "inputs"

if on_existing == "overwrite" and pred_out.exists():
  shutil.rmtree(pred_out)
inputs_dir.mkdir(parents=True, exist_ok=True)

# 2. Resolve glycans + covalent bonds from PREP_STATE, plus any extra non-covalent ligands
glycan_chains = {}
covalent_bonds = []
if include_glycans:
  if "PREP_STATE" in globals() and PREP_STATE.get("glycan_chains"):
    glycan_chains = PREP_STATE["glycan_chains"]      # {chain_id: [(resnum, ccd), ...]}
    covalent_bonds = PREP_STATE.get("covalent_bonds", [])
  else:
    print("Warning: include_glycans=True but PREP_STATE has no glycan_chains. Re-run Cell 2 if glycans should be included.", flush=True)

def _split_colons(s):
  return [tok.strip() for tok in re.sub(r":+", ":", s).strip(":").split(":") if tok.strip()]

ccd_codes = [c.upper() for c in _split_colons(ligand_ccd)]
smiles_list = _split_colons(ligand_smiles)  # case-sensitive

def _fetch_smiles_from_rcsb(ccd: str) -> str:
  url = f"https://data.rcsb.org/rest/v1/core/chemcomp/{ccd}"
  with urllib.request.urlopen(url, timeout=15) as resp:
    data = json.loads(resp.read().decode())
  descs = data.get("rcsb_chem_comp_descriptor", {})
  return descs.get("smilesstereo") or descs.get("smiles") or ""

if ccd_codes:
  fetched_smiles = {}
  for code in ccd_codes:
    try:
      smi = _fetch_smiles_from_rcsb(code)
      if smi:
        fetched_smiles[code] = smi
        print(f"RCSB lookup — {code} SMILES: {smi}", flush=True)
    except Exception as e:
      print(f"Warning: could not fetch SMILES for {code} from RCSB ({e})", flush=True)

  if use_smiles_from_ccd:
    for code in ccd_codes:
      if code not in fetched_smiles:
        raise RuntimeError(f"Could not resolve SMILES for CCD '{code}'. Uncheck use_smiles_from_ccd or paste SMILES manually.")
      smiles_list.append(fetched_smiles[code])
    ccd_codes = []

# 3. Collect multi-chain designed sequences from MPNN FASTAs
fasta_files = sorted((mpnn_path / "seqs").glob("*.fa*")) if (mpnn_path / "seqs").is_dir() else []
if not fasta_files:
  fasta_files = sorted(mpnn_path.rglob("*.fa*"))
if not fasta_files:
  raise FileNotFoundError(f"No .fa/.fasta files found in {mpnn_path}")

def _parse_fasta(path: pathlib.Path) -> list[tuple[str, str]]:
  entries, header, chunks = [], None, []
  for raw in path.read_text().splitlines():
    line = raw.strip()
    if not line:
      continue
    if line.startswith(">"):
      if header is not None:
        entries.append((header, "".join(chunks)))
      header, chunks = line[1:], []
    else:
      chunks.append(line)
  if header is not None:
    entries.append((header, "".join(chunks)))
  return entries

# Target chain length from Cell 2 (used to split concatenated Foundry MPNN FASTAs)
target_len = len(PREP_STATE["res_map"]) if ("PREP_STATE" in globals() and PREP_STATE.get("res_map")) else 0

designs = []  # list of (stem, [chain_seq_A, chain_seq_B, ...])
for fa in fasta_files:
  entries = _parse_fasta(fa)
  if not entries:
    continue
  has_native_entry0 = "b0_d0" not in entries[0][0]
  idx = min(seq_index if has_native_entry0 else max(0, seq_index - 1), len(entries) - 1)
  raw_seq = entries[idx][1].strip().upper()

  if ":" in raw_seq or "/" in raw_seq:
    chain_seqs = [c.strip() for c in re.split(r"[:/]", raw_seq) if c.strip()]
  elif target_len > 0 and len(raw_seq) > target_len:
    chain_seqs = [raw_seq[:-target_len], raw_seq[-target_len:]]
  else:
    chain_seqs = [raw_seq]

  designs.append((fa.stem, chain_seqs))

# 4. Pre-fetch MMseqs2 MSAs once per unique sequence (shared across AF3, OpenFold3, Boltz-2, and RF3)
def _fetch_mmseqs2_a3m(seq: str, out_a3m: pathlib.Path) -> pathlib.Path:
  """Query the ColabFold MMseqs2 API once for a unique protein sequence and save uniref.a3m."""
  host = "https://api.colabfold.com"
  headers = {"User-Agent": "colabfold/foundry-notebook"}

  data = urllib.parse.urlencode({"q": f">101\n{seq}\n", "mode": "env"}).encode()
  req = urllib.request.Request(f"{host}/ticket/msa", data=data, headers=headers)
  with urllib.request.urlopen(req, timeout=30) as resp:
    ticket = json.loads(resp.read().decode())
  job_id = ticket["id"]

  while ticket.get("status") in ("UNKNOWN", "RUNNING", "PENDING"):
    time.sleep(5)
    req = urllib.request.Request(f"{host}/ticket/{job_id}", headers=headers)
    with urllib.request.urlopen(req, timeout=30) as resp:
      ticket = json.loads(resp.read().decode())
  if ticket.get("status") != "COMPLETE":
    raise RuntimeError(f"MMseqs2 server failed for job {job_id}: {ticket}")

  req = urllib.request.Request(f"{host}/result/download/{job_id}", headers=headers)
  with urllib.request.urlopen(req, timeout=60) as resp:
    tar_bytes = resp.read()
  with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode="r:gz") as tar:
    a3m_members = [m for m in tar.getmembers() if m.name.endswith(".a3m")]
    member = next((m for m in a3m_members if "uniref.a3m" in m.name), a3m_members[0])
    a3m_text = tar.extractfile(member).read().decode("utf-8", errors="replace").replace("\x00", "")
    out_a3m.write_text(a3m_text)
  return out_a3m

CHAIN_IDS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
msa_cache: dict[str, pathlib.Path] = {}  # unique seq -> cached .a3m path

if msa_mode != "single_sequence":
  for _, chain_seqs in designs:
    for c_idx, seq in enumerate(chain_seqs):
      need_msa = (msa_mode == "mmseqs2_server") or (msa_mode == "binder_single_target_msa" and c_idx > 0)
      if need_msa and seq not in msa_cache:
        a3m_path = inputs_dir / f"msa_seq{len(msa_cache)}.a3m"
        print(f"Fetching MMseqs2 MSA once for unique chain {CHAIN_IDS[c_idx]} ({len(seq)} aa) -> {a3m_path.name}...", flush=True)
        msa_cache[seq] = _fetch_mmseqs2_a3m(seq, a3m_path)

# Helper for Boltz-2 / RF3: expand multi-residue glycan chains into single-CCD monosaccharide chains
def _expand_monosaccharide_chains(used_protein_cids: set[str]):
  """Map each (orig_gly_chain, resnum) -> unique single-char chain ID for single-CCD ligand models."""
  used = set(used_protein_cids) | set(glycan_chains.keys())
  free_cids = [c for c in CHAIN_IDS if c not in used]
  mono_entries = []  # [(mapped_cid, ccd_code), ...]
  mono_map = {}      # (orig_cid, resnum) -> mapped_cid
  for orig_cid, res_list in glycan_chains.items():
    for idx, (rnum, rname) in enumerate(res_list):
      mapped_cid = orig_cid if idx == 0 else free_cids.pop(0)
      mono_map[(orig_cid, rnum)] = mapped_cid
      mono_entries.append((mapped_cid, rname))
  return mono_entries, mono_map, free_cids

seed_list = sorted({int(s) for s in re.findall(r"\d+", seeds)}) or [1]
n_prot = len(designs[0][1]) if designs else 0
n_gly_res = sum(len(v) for v in glycan_chains.values())
print(
    f"Prepared {len(designs)} complex(es) ({n_prot} protein chain(s), "
    f"{len(glycan_chains)} glycan chain(s) / {n_gly_res} sugar(s), {len(covalent_bonds)} covalent bond(s), "
    f"{len(ccd_codes)} extra CCD, {len(smiles_list)} SMILES, {len(msa_cache)} cached MSA(s)) "
    f"for {selected} [{msa_mode}] -> {inputs_dir}",
    flush=True,
)

def _emit_terminal_script(env_label: str, exports: dict[str, str], cmd_args: list[str], script_name: str):
  env_path = pathlib.Path(os.environ["ENVS_DIR"]) / env_label
  lines = [
      "#!/usr/bin/env bash",
      "set -euo pipefail",
      "",
      f"# Activate '{env_label}' environment & set required paths",
      f"export VIRTUAL_ENV={shlex.quote(str(env_path))}",
      f'export PATH="{env_path}/bin:$PATH"',
  ]
  for k, v in exports.items():
    lines.append(f"export {k}={shlex.quote(str(v))}")
  lines.append("")
  lines.append(f"# Run {selected}")
  lines.append(" \\\n  ".join(shlex.quote(str(c)) for c in cmd_args))
  script = "\n".join(lines) + "\n"

  sh_path = pred_out.parent / script_name
  sh_path.write_text(script)
  sh_path.chmod(0o755)
  print("\n" + "=" * 72)
  print("TERMINAL INSTRUCTIONS — Copy & paste into a terminal, or run:")
  print(f"  bash {sh_path}")
  print("=" * 72)
  print(script)

# 5. AlphaFold3 / OpenFold3
if selected in ("AlphaFold3", "OpenFold3"):
  all_af3_ccds = sorted(set(ccd_codes) | {rname for res_list in glycan_chains.values() for _, rname in res_list})
  if all_af3_ccds:
    extra_str = json.dumps(all_af3_ccds)
    ccd_code = (
        f"import os, importlib.metadata as md; "
        f"from alphafold3.constants import ccd_fetch; "
        f"root = os.path.dirname(md.distribution('alphafold3-colabfold').locate_file('alphafold3')); "
        f"conv = os.path.join(root, 'alphafold3', 'constants', 'converters'); "
        f"os.makedirs(conv, exist_ok=True); "
        f"ccd_fetch.write_pickles(ccd_fetch.codes_for_input(extra={extra_str}), "
        f"os.path.join(conv, 'ccd.pickle'), "
        f"os.path.join(conv, 'chemical_component_sets.pickle'), "
        f"libcifpp_dir=os.path.join(root, 'share', 'libcifpp'))"
    )
    print(f"Fetching CCD definitions for {all_af3_ccds}...", flush=True)
    run_command_with_env("af3", ["python", "-c", ccd_code], quiet=True)

  msa_text_cache = {seq: p.read_text() for seq, p in msa_cache.items()}
  written_jsons, first_preview = [], None

  for stem, chain_seqs in designs:
    job_name = stem.lower()
    entities = []
    used_cids = set()
    for c_idx, seq in enumerate(chain_seqs):
      cid = CHAIN_IDS[c_idx]
      used_cids.add(cid)
      need_msa = (msa_mode == "mmseqs2_server") or (msa_mode == "binder_single_target_msa" and c_idx > 0)
      unpaired = msa_text_cache[seq] if (need_msa and seq in msa_text_cache) else f">query\n{seq}\n"
      entities.append({
          "protein": {
              "id": cid,
              "sequence": seq,
              "unpairedMsa": unpaired,
              "pairedMsa": "",
              "templates": [],
          }
      })

    # Add glycan chains (keeping exact Cell 2 chain IDs C, D, E... and multi-CCD lists like ["NAG", "NAG"])
    for g_cid, res_list in glycan_chains.items():
      used_cids.add(g_cid)
      entities.append({"ligand": {"id": g_cid, "ccdCodes": [rname for _, rname in res_list]}})

    # Add any extra non-covalent ligands on remaining free chain IDs
    free_cids = [c for c in CHAIN_IDS if c not in used_cids]
    for code in ccd_codes:
      entities.append({"ligand": {"id": free_cids.pop(0), "ccdCodes": [code]}})
    for smi in smiles_list:
      entities.append({"ligand": {"id": free_cids.pop(0), "smiles": smi}})

    fold_input = {
        "name": job_name,
        "sequences": entities,
        "modelSeeds": seed_list,
        "dialect": "alphafold3",
        "version": 1,
    }
    if covalent_bonds:
      fold_input["bondedAtomPairs"] = [
          [[b["chain1"], int(b["resnum1"]), b["atom1"]], [b["chain2"], int(b["resnum2"]), b["atom2"]]]
          for b in covalent_bonds
      ]

    json_path = inputs_dir / f"{job_name}.json"
    json_path.write_text(json.dumps(fold_input, indent=2) + "\n")
    written_jsons.append(json_path)

    if first_preview is None:
      prev = copy.deepcopy(fold_input)
      for item in prev["sequences"]:
        if "protein" in item:
          msa_s = item["protein"].get("unpairedMsa", "")
          n_seqs = msa_s.count("\n>") + (1 if msa_s.startswith(">") else 0)
          if n_seqs > 1:
            item["protein"]["unpairedMsa"] = f"<cached MMseqs2 MSA: {n_seqs} sequences>"
      first_preview = prev

  if written_jsons:
    extra = f" (showing 1 of {len(written_jsons)})" if len(written_jsons) > 1 else ""
    print(f"\nInput JSON saved to {written_jsons[0]}{extra}:")
    print(json.dumps(first_preview, indent=2), flush=True)

  cap_out = subprocess.run(
      ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
      capture_output=True, text=True,
  ).stdout.split()
  cap = float(cap_out[0]) if cap_out else 0.0

  if cap < 8.0:
    flash_impl, xla_flag = "xla", "--xla_disable_hlo_passes=custom-kernel-fusion-rewriter"
  elif 8.0 < cap < 9.0:
    flash_impl, xla_flag = "xla", "--xla_gpu_enable_triton_gemm=false"
  else:
    flash_impl, xla_flag = "triton", "--xla_gpu_enable_triton_gemm=false"

  xla_flags = os.environ.get("XLA_FLAGS", "")
  if xla_flag not in xla_flags:
    xla_flags = f"{xla_flags} {xla_flag}".strip()
  os.environ["XLA_FLAGS"] = xla_flags

  for var in ("XLA_PYTHON_CLIENT_PREALLOCATE", "XLA_PYTHON_CLIENT_ALLOCATOR", "TF_FORCE_UNIFIED_MEMORY"):
    os.environ.pop(var, None)

  af3_bin = pathlib.Path(os.environ["ENVS_DIR"]) / "af3" / "bin"
  cmd = [
      str(af3_bin / "python"), f"{os.environ['AF3_DIR']}/run_alphafold.py",
      "--stderrthreshold=error",
      f"--input_dir={inputs_dir}",
      f"--model={model_key}",
      "--norun_data_pipeline",
      f"--output_dir={pred_out}",
      f"--cache_dir={os.environ['AF3_CACHE_DIR']}",
      "--force_output_dir",
      f"--flash_attention_implementation={flash_impl}",
      f"--num_recycles={max(1, int(num_recycles))}",
      f"--num_diffusion_samples={max(1, int(num_diffusion_samples))}",
  ]
  if selected == "AlphaFold3":
    cmd.append(f"--model_dir={os.environ['AF3_WEIGHTS_DIR']}")

  if execution_mode == "Prepare inputs & print terminal commands":
    _emit_terminal_script(
        "af3",
        {
            "XLA_FLAGS": xla_flags,
            "TF_CPP_MIN_LOG_LEVEL": "3",
            "ABSL_MIN_LOG_LEVEL": "2",
            "GLOG_minloglevel": "2",
        },
        ["python", *cmd[1:]],
        f"run_{model_key}.sh",
    )
  else:
    print(f"[af3] {' '.join(cmd)}", flush=True)
    env = os.environ.copy()
    env["PATH"] = f"{af3_bin}:{env.get('PATH', '')}"
    env["TF_CPP_MIN_LOG_LEVEL"] = "3"
    env["ABSL_MIN_LOG_LEVEL"] = "2"
    env["GLOG_minloglevel"] = "2"
    env["PYTHONUNBUFFERED"] = "1"

    noise = ("cuda_timer.cc:", "TOKAMAX autotune unavailable")
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env
    ) as proc:
      skip_tb = False
      for line in proc.stdout:
        if any(k in line for k in noise):
          skip_tb = "TOKAMAX" in line
          continue
        if skip_tb:
          if line.startswith(("  File ", "    ", "Traceback ", "AttributeError:", "RuntimeError:")):
            continue
          skip_tb = False
        print(line, end="", flush=True)

      if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)

# 6. Boltz-2
elif selected == "Boltz2":
  written_yamls = []
  for stem, chain_seqs in designs:
    lines = ["version: 1", "sequences:"]
    prot_cids = set()
    for c_idx, seq in enumerate(chain_seqs):
      cid = CHAIN_IDS[c_idx]
      prot_cids.add(cid)
      lines.append("- protein:")
      lines.append(f"    id: {cid}")
      lines.append(f"    sequence: {seq}")
      need_msa = (msa_mode == "mmseqs2_server") or (msa_mode == "binder_single_target_msa" and c_idx > 0)
      if need_msa and seq in msa_cache:
        lines.append(f"    msa: {msa_cache[seq]}")
      else:
        lines.append("    msa: empty")

    # Add each monosaccharide as a single-CCD ligand chain and map covalent bonds
    mono_entries, mono_map, free_cids = _expand_monosaccharide_chains(prot_cids)
    for mapped_cid, rname in mono_entries:
      lines.append("- ligand:")
      lines.append(f"    id: {mapped_cid}")
      lines.append(f"    ccd: {rname}")

    for code in ccd_codes:
      lines.append("- ligand:")
      lines.append(f"    id: {free_cids.pop(0)}")
      lines.append(f"    ccd: {code}")
    for smi in smiles_list:
      lines.append("- ligand:")
      lines.append(f"    id: {free_cids.pop(0)}")
      lines.append(f"    smiles: {json.dumps(smi)}")

    if covalent_bonds:
      lines.append("constraints:")
      for b in covalent_bonds:
        c1 = mono_map.get((b["chain1"], b["resnum1"]), b["chain1"])
        r1 = 1 if (b["chain1"], b["resnum1"]) in mono_map else int(b["resnum1"])
        c2 = mono_map.get((b["chain2"], b["resnum2"]), b["chain2"])
        r2 = 1 if (b["chain2"], b["resnum2"]) in mono_map else int(b["resnum2"])
        lines.append("- bond:")
        lines.append(f"    atom1: [{c1}, {r1}, {b['atom1']}]")
        lines.append(f"    atom2: [{c2}, {r2}, {b['atom2']}]")

    yaml_path = inputs_dir / f"{stem}.yaml"
    yaml_path.write_text("\n".join(lines) + "\n")
    written_yamls.append(yaml_path)

  if written_yamls:
    extra = f" (showing 1 of {len(written_yamls)})" if len(written_yamls) > 1 else ""
    print(f"\nInput YAML saved to {written_yamls[0]}{extra}:")
    print(written_yamls[0].read_text(), flush=True)

  cmd = [
      "boltz", "predict", str(inputs_dir),
      "--cache", os.environ["BOLTZ_CACHE"],
      "--out_dir", str(pred_out),
      "--recycling_steps", str(max(1, int(num_recycles))),
      "--sampling_steps", str(max(1, int(boltz_sampling_steps))),
      "--diffusion_samples", str(max(1, int(num_diffusion_samples))),
      "--step_scale", str(float(boltz_step_scale)),
      "--output_format", boltz_output_format,
  ]
  if boltz_use_potentials:
    cmd.append("--use_potentials")
  if on_existing == "overwrite":
    cmd.append("--override")

  if execution_mode == "Prepare inputs & print terminal commands":
    _emit_terminal_script(
        "boltz",
        {"BOLTZ_CACHE": os.environ["BOLTZ_CACHE"]},
        cmd,
        "run_boltz2.sh",
    )
  else:
    print(f"[boltz] {' '.join(cmd)}", flush=True)
    boltz_bin = pathlib.Path(os.environ["ENVS_DIR"]) / "boltz" / "bin"
    cmd[0] = str(boltz_bin / "boltz")

    env = os.environ.copy()
    env["PATH"] = f"{boltz_bin}:{env.get('PATH', '')}"

    log_path = pred_out / "boltz.log"
    ext = "pdb" if boltz_output_format == "pdb" else "cif"
    seen, total, t0 = set(), len(designs), time.time()

    print(f"Running Boltz-2 on {total} complex(es) (log: {log_path.name})...", flush=True)
    with open(log_path, "w") as log_f:
      proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, env=env)
      while proc.poll() is None:
        for p in sorted(pred_out.rglob(f"*_model_0.{ext}")):
          if not p.is_file():
            continue
          name = p.parent.name
          if name not in seen:
            seen.add(name)
            print(f"  [{len(seen)}/{total}] {name} ({time.time() - t0:.0f}s elapsed)", flush=True)
        time.sleep(2)

    for p in sorted(pred_out.rglob(f"*_model_0.{ext}")):
      if not p.is_file():
        continue
      name = p.parent.name
      if name not in seen:
        seen.add(name)
        print(f"  [{len(seen)}/{total}] {name} ({time.time() - t0:.0f}s elapsed)", flush=True)

    if proc.returncode != 0:
      print(log_path.read_text()[-3000:])
      raise subprocess.CalledProcessError(proc.returncode, cmd)

# 7. RosettaFold3
elif selected == "RosettaFold3":
  rf3_entries = []
  for stem, chain_seqs in designs:
    components = []
    prot_cids = set()
    for c_idx, seq in enumerate(chain_seqs):
      cid = CHAIN_IDS[c_idx]
      prot_cids.add(cid)
      comp = {"seq": seq, "chain_id": cid}
      need_msa = (msa_mode == "mmseqs2_server") or (msa_mode == "binder_single_target_msa" and c_idx > 0)
      if need_msa and seq in msa_cache:
        comp["msa_path"] = str(msa_cache[seq])
      components.append(comp)

    mono_entries, mono_map, free_cids = _expand_monosaccharide_chains(prot_cids)
    for mapped_cid, rname in mono_entries:
      components.append({"ccd_code": rname, "chain_id": mapped_cid})

    for code in ccd_codes:
      components.append({"ccd_code": code, "chain_id": free_cids.pop(0)})
    for smi in smiles_list:
      components.append({"smiles": smi, "chain_id": free_cids.pop(0)})

    entry = {"name": stem, "components": components}
    if covalent_bonds:
      bonds_list = []
      for b in covalent_bonds:
        c1 = mono_map.get((b["chain1"], b["resnum1"]), b["chain1"])
        r1 = 1 if (b["chain1"], b["resnum1"]) in mono_map else int(b["resnum1"])
        c2 = mono_map.get((b["chain2"], b["resnum2"]), b["chain2"])
        r2 = 1 if (b["chain2"], b["resnum2"]) in mono_map else int(b["resnum2"])
        bonds_list.append([
            f"{c1}/{b['resname1']}/{r1}/{b['atom1']}",
            f"{c2}/{b['resname2']}/{r2}/{b['atom2']}",
        ])
      entry["bonds"] = bonds_list
    rf3_entries.append(entry)

  rf3_json = inputs_dir / "rf3_inputs.json"
  rf3_json.write_text(json.dumps(rf3_entries, indent=2) + "\n")

  print(f"\nInput JSON saved to {rf3_json} ({len(rf3_entries)} complex(es)):")
  print(rf3_json.read_text(), flush=True)

  cmd = [
      "rf3", "fold",
      f"inputs={rf3_json}",
      f"out_dir={pred_out}",
      f"n_recycles={max(1, int(num_recycles))}",
      f"diffusion_batch_size={max(1, int(num_diffusion_samples))}",
      f"num_steps={max(1, int(rf3_num_steps))}",
      f"early_stopping_plddt_threshold={float(rf3_early_stopping_plddt)}",
      f"seed={seed_list[0]}",
  ]
  if on_existing == "skip":
    cmd.append("skip_existing=True")

  if execution_mode == "Prepare inputs & print terminal commands":
    _emit_terminal_script(
        "foundry",
        {
            "FOUNDRY_CHECKPOINT_DIRS": os.environ["FOUNDRY_CHECKPOINT_DIRS"],
            "CCD_MIRROR_PATH": "/tmp",
            "PDB_MIRROR_PATH": "/tmp",
            "PYTHONWARNINGS": "ignore",
        },
        cmd,
        "run_rosettafold3.sh",
    )
  else:
    run_command_with_env("foundry", cmd)

if execution_mode == "Run in Colab":
  print(f"\nDone! Predictions saved to: {pred_out}", flush=True)

In [ ]:
#@title Visualize RFD3 Designs vs. Predicted Structures
#@markdown Interactively compare each RFD3 backbone (grey) against its refolded prediction(s) with Cα RMSD.

design_glob = "8S6Z_binder_design/rfd3_out/*.cif.gz" #@param {type:"string"}
#@markdown Leave `pred_dir` blank to automatically use `pred_out` from the refolding cell above, or enter a custom path.
pred_dir = "" #@param {type:"string"}
#@markdown Show only the top-ranked model per design (`True`) or superimpose up to `max_models` samples (`False`).
only_best = True #@param {type:"boolean"}
max_models = 5 #@param {type:"integer"}
#@markdown Set the size of the visualization
width = 700 #@param {type:"integer"}
height = 500 #@param {type:"integer"}

target_pred = pred_dir.strip() or (str(pred_out) if "pred_out" in globals() else "")
if not target_pred:
  raise ValueError("No prediction folder specified. Run the refolding cell first or enter a path in pred_dir.")

browse_designs(
    design_glob=design_glob,
    pred_root=target_pred,
    only_best=only_best,
    max_models=max_models,
    width=width,
    height=height,
)

In [ ]:
#@title Quick sanity check that glycans are bonded

import pathlib, numpy as np

pred_files = [
    p for p in sorted(pathlib.Path(pred_out).rglob("*"))
    if p.is_file()
    and "inputs" not in p.parts
    and p.name.endswith(("_model.cif", "_model.cif.gz", "_model_0.cif", "_model_0.pdb"))
]
struct = _load(pred_files[0])
print(f"Checking: {pred_files[0]}\n")

is_prot = struc.filter_amino_acids(struct)
nags = struct[_ligand_mask(struct)]

for b in PREP_STATE["covalent_bonds"]:
  if b["chain1"] != "B":
    continue
  asn_nd2 = struct[is_prot & (struct.chain_id == "B") & (struct.res_id == b["resnum1"]) & (struct.atom_name == b["atom1"])]
  nag_c1s = nags[nags.atom_name == b["atom2"]]
  if len(asn_nd2) and len(nag_c1s):
    dists = np.linalg.norm(nag_c1s.coord - asn_nd2.coord[0], axis=-1)
    idx = int(np.argmin(dists))
    hit = nag_c1s[idx]
    status = "BONDED" if dists[idx] < 2.0 else "DETACHED"
    print(f"  B{b['resnum1']}({b['resname1']}).{b['atom1']} -> nearest {hit.chain_id}{hit.res_id}({hit.res_name}).{hit.atom_name}: {dists[idx]:.2f} Å  [{status}]")

In [ ]:
#@title Evaluate & Plot Binder Design Metrics (AF3 / OpenFold3 / Boltz-2 / RF3 + HBPLUS + ipSAE)
#@markdown Leave `pred_root` blank to automatically use `pred_out` from the refolding cell above, or enter a custom path.
design_glob = "8S6Z_binder_design/rfd3_out/*.cif.gz" #@param {type:"string"}
pred_root = "" #@param {type:"string"}
only_best = True #@param {type:"boolean"}
out_prefix = "8S6Z_binder_design/binder_eval" #@param {type:"string"}

#@markdown ### Hit Thresholds (for dashed guide lines & pass/fail coloring)
pose_rmsd_cutoff = 5.0 #@param {type:"number"}
iptm_cutoff = 0.75 #@param {type:"number"}
ipsae_cutoff = 0.60 #@param {type:"number"}
min_pae_cutoff = 5.0 #@param {type:"number"}
plddt_cutoff = 80.0 #@param {type:"number"}
ipsae_pae_cutoff = 10.0 #@param {type:"number"}

import glob, gzip, json, os, pathlib, re, subprocess, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import biotite.structure as struc
import biotite.structure.io.pdb as pdb_io
import biotite.structure.io.pdbx as pdbx_io


def _load_struct(path):
  p = str(path)
  opener = gzip.open if p.endswith(".gz") else open
  base = p[:-3] if p.endswith(".gz") else p
  if base.endswith(".cif"):
    with opener(p, "rt") as f:
      struct = pdbx_io.get_structure(pdbx_io.CIFFile.read(f), model=1, extra_fields=["b_factor"])
  else:
    with opener(p, "rt") as f:
      struct = pdb_io.PDBFile.read(f).get_structure(model=1, extra_fields=["b_factor"])

  finite_mask = np.all(np.isfinite(struct.coord), axis=-1)
  return struct[finite_mask] if not np.all(finite_mask) else struct


def _atom_elements(struct):
  """Return uppercase stripped element symbols, falling back to atom_name if missing."""
  elems = (
      np.char.strip(np.char.upper(struct.element.astype(str)))
      if hasattr(struct, "element")
      else np.full(len(struct), "", dtype=object)
  )
  missing = np.isin(elems, ["", "?", "."])
  if np.any(missing):
    is_prot = struc.filter_amino_acids(struct)
    names = np.char.strip(np.char.upper(struct.atom_name.astype(str)))
    two_letter = {"CL", "BR", "NA", "MG", "ZN", "FE", "CA", "MN", "CU", "CO", "NI", "SE"}
    elems = elems.astype(object)
    for idx in np.where(missing)[0]:
      m = re.match(r"^[0-9]*([A-Z]+)", str(names[idx]))
      if not m:
        elems[idx] = ""
      else:
        letters = m.group(1)
        elems[idx] = letters[:2] if (not is_prot[idx] and letters[:2] in two_letter) else letters[0]
    elems = elems.astype(str)
  return elems


def _ligand_mask(struct):
  """Select non-polymer, non-solvent heavy atoms (captures glycans, small molecules, and ions)."""
  is_polymer = struc.filter_amino_acids(struct) | struc.filter_nucleotides(struct)
  elem = _atom_elements(struct)
  return (~is_polymer) & (~struc.filter_solvent(struct)) & (elem != "H")


def _superimpose_ca_eval(fixed_ca, mobile_ca):
  if len(fixed_ca) == len(mobile_ca) and len(fixed_ca) >= 3:
    res = struc.superimpose(fixed_ca, mobile_ca)
    tf = res[1] if isinstance(res, tuple) else res
    rmsd = float(struc.rmsd(fixed_ca, tf.apply(mobile_ca)))
    return tf, rmsd, len(fixed_ca)
  res = struc.superimpose_structural_homologs(fixed_ca, mobile_ca)
  tf, f_idx, m_idx = res[1:] if len(res) == 4 else res
  rmsd = float(struc.rmsd(fixed_ca[f_idx], tf.apply(mobile_ca[m_idx])))
  return tf, rmsd, len(f_idx)


def _get_het_residues(struct):
  het = struct[_ligand_mask(struct)]
  if len(het) == 0:
    return []
  residues = []
  for cid in dict.fromkeys(het.chain_id.astype(str)):
    ch = het[het.chain_id.astype(str) == cid]
    for rid in dict.fromkeys(ch.res_id):
      res = ch[ch.res_id == rid]
      if len(res) >= 3:
        rname = str(res.res_name[0]).strip().upper()
        centroid = np.mean(res.coord, axis=0)
        amap = {str(a.atom_name).strip(): a.coord for a in res}
        residues.append((rname, centroid, amap))
  return residues


def _ligand_rmsd_val(fixed, aligned_mobile):
  """Pooled heavy-atom RMSD across all matched glycan/ligand residues after target alignment."""
  f_res = _get_het_residues(fixed)
  m_res = _get_het_residues(aligned_mobile)
  if not f_res or not m_res:
    return np.nan

  pairs = []
  for i, (fn, fc, _) in enumerate(f_res):
    for j, (mn, mc, _) in enumerate(m_res):
      dist = float(np.linalg.norm(fc - mc))
      pairs.append((0 if fn == mn else 1, dist, i, j))
  pairs.sort(key=lambda x: (x[0], x[1]))

  used_f, used_m, matched = set(), set(), []
  for _, dist, i, j in pairs:
    if i not in used_f and j not in used_m:
      used_f.add(i)
      used_m.add(j)
      matched.append((i, j, dist))

  if not matched:
    return np.nan

  sq_dists = []
  for i, j, _ in matched:
    f_map, m_map = f_res[i][2], m_res[j][2]
    common = sorted(set(f_map) & set(m_map))
    if len(common) >= 3:
      fc = np.array([f_map[k] for k in common])
      mc = np.array([m_map[k] for k in common])
      sq_dists.append(np.sum((fc - mc) ** 2, axis=-1))

  if sq_dists:
    return float(np.sqrt(np.mean(np.concatenate(sq_dists))))
  return float(np.sqrt(np.mean([d ** 2 for _, _, d in matched])))


def _calc_rmsds(fixed, mobile):
  f_ca = fixed[(fixed.atom_name == "CA") & struc.filter_amino_acids(fixed)]
  m_ca = mobile[(mobile.atom_name == "CA") & struc.filter_amino_acids(mobile)]

  out = {"target_rmsd": np.nan, "binder_pose_rmsd": np.nan, "binder_fold_rmsd": np.nan, "ligand_rmsd": np.nan}
  if "B" in set(f_ca.chain_id) and "B" in set(m_ca.chain_id):
    f_b, m_b = f_ca[f_ca.chain_id == "B"], m_ca[m_ca.chain_id == "B"]
    tf_b, b_rmsd, _ = _superimpose_ca_eval(f_b, m_b)
    aligned = tf_b.apply(mobile)
    out["target_rmsd"] = b_rmsd

    if "A" in set(f_ca.chain_id) and "A" in set(m_ca.chain_id):
      f_a = f_ca[f_ca.chain_id == "A"]
      m_a_pose = aligned[(aligned.atom_name == "CA") & struc.filter_amino_acids(aligned) & (aligned.chain_id == "A")]
      if len(f_a) == len(m_a_pose):
        out["binder_pose_rmsd"] = float(struc.rmsd(f_a, m_a_pose))
      else:
        f_map = {int(r): c for r, c in zip(f_a.res_id, f_a.coord)}
        m_map = {int(r): c for r, c in zip(m_a_pose.res_id, m_a_pose.coord)}
        common = sorted(set(f_map) & set(m_map))
        if len(common) >= 3:
          fc = np.array([f_map[k] for k in common])
          mc = np.array([m_map[k] for k in common])
          out["binder_pose_rmsd"] = float(np.sqrt(np.mean(np.sum((fc - mc) ** 2, axis=-1))))
      _, fold_rmsd, _ = _superimpose_ca_eval(f_a, m_ca[m_ca.chain_id == "A"])
      out["binder_fold_rmsd"] = fold_rmsd

    out["ligand_rmsd"] = _ligand_rmsd_val(fixed, aligned)
  else:
    _, fold_rmsd, _ = _superimpose_ca_eval(f_ca, m_ca)
    out["binder_fold_rmsd"] = fold_rmsd
  return out


def _binder_plddt(mobile):
  a_ca = mobile[(mobile.atom_name == "CA") & struc.filter_amino_acids(mobile) & (mobile.chain_id == "A")]
  if len(a_ca) == 0 or not hasattr(a_ca, "b_factor"):
    return np.nan
  vals = np.asarray(a_ca.b_factor, dtype=float)
  return float(np.mean(vals) * (100.0 if np.nanmax(vals) <= 1.0 else 1.0))


def _load_cond_map(design_path):
  """Read select_hbond_donor / select_hbond_acceptor from the RFD3 companion .json."""
  p = pathlib.Path(design_path)
  stem = p.name[:-7] if p.name.lower().endswith(".cif.gz") else p.stem
  meta = p.with_name(f"{stem}.json")
  if not meta.exists():
    return {}
  spec = json.loads(meta.read_text()).get("specification", {})
  cond_map = {}
  for key in ("select_hbond_donor", "select_hbond_acceptor"):
    raw = spec.get(key)
    if not raw:
      continue
    items = {t.strip(): "ALL" for t in raw.replace(";", ",").split(",") if t.strip()} if isinstance(raw, str) else raw
    for res_key, atoms in items.items():
      m = re.match(r"^([A-Za-z0-9])(\d+)$", str(res_key).strip())
      if not m:
        continue
      rk = (m.group(1), int(m.group(2)))
      if atoms is None or str(atoms).strip().upper() in ("", "ALL", "*"):
        cond_map[rk] = None
      else:
        alist = atoms if isinstance(atoms, (list, tuple, set)) else str(atoms).split(",")
        aset = {str(a).strip().upper() for a in alist if str(a).strip()}
        cond_map[rk] = (cond_map[rk] | aset) if (rk in cond_map and cond_map[rk] is not None) else aset
  return cond_map


def _count_hbonds(struct, cond_map=None, binder_chain="A"):
  """Counts deduplicated HBPLUS interface H-bonds (A–B, A–glycans/ligand, and conditioned hits)."""
  hbplus_bin = os.environ.get("HBPLUS_PATH", "/usr/local/bin/hbplus")
  if not (hbplus_bin and os.access(hbplus_bin, os.X_OK)):
    raise FileNotFoundError(f"HBPLUS executable not found at {hbplus_bin}")

  lig_mask = _ligand_mask(struct)
  lig_res = set(zip(struct.chain_id[lig_mask].astype(str), struct.res_id[lig_mask].astype(int)))

  with tempfile.TemporaryDirectory() as tmp:
    pdb_path = pathlib.Path(tmp) / "struct.pdb"
    pdb_file = pdb_io.PDBFile()
    pdb_file.set_structure(struct)
    pdb_file.write(str(pdb_path))
    subprocess.run([hbplus_bin, str(pdb_path)], cwd=tmp, capture_output=True, text=True, check=True)
    hb2 = pathlib.Path(tmp) / "struct.hb2"
    lines = hb2.read_text().splitlines()[8:] if hb2.exists() else []

  seen = set()
  n_ab, n_ac, n_cond = 0, 0, 0
  for line in lines:
    if len(line) < 35:
      continue
    try:
      d_ch, d_res, d_atom = line[0].strip(), int(line[1:5]), line[10:14].strip().upper()
      a_ch, a_res, a_atom = line[14].strip(), int(line[15:19]), line[24:28].strip().upper()
    except ValueError:
      continue

    if (d_ch == binder_chain) == (a_ch == binder_chain):
      continue

    pair_key = frozenset([(d_ch, d_res, d_atom), (a_ch, a_res, a_atom)])
    if pair_key in seen:
      continue
    seen.add(pair_key)

    t_ch, t_res, t_atom = (a_ch, a_res, a_atom) if d_ch == binder_chain else (d_ch, d_res, d_atom)
    if (t_ch, t_res) in lig_res:
      n_ac += 1
    else:
      n_ab += 1

    if cond_map and (t_ch, t_res) in cond_map:
      allowed = cond_map[(t_ch, t_res)]
      if allowed is None or t_atom in allowed:
        n_cond += 1

  return {"hbonds_AB": n_ab, "hbonds_AC": n_ac, "cond_hbonds": n_cond}


def _calc_ipsae_pair(pae, mask_i, mask_j, pae_cutoff=10.0):
  """Symmetric Dunbrack ipSAE (d0_res) between two token masks."""
  if not np.any(mask_i) or not np.any(mask_j):
    return np.nan

  def _directional(sub):
    valid = sub < pae_cutoff
    n_valid = valid.sum(axis=1)
    if not np.any(n_valid > 0):
      return 0.0
    d0 = np.maximum(1.0, 1.24 * np.cbrt(np.maximum(n_valid, 27) - 15.0) - 1.8)
    scores = np.where(valid, 1.0 / (1.0 + (sub / d0[:, None]) ** 2), 0.0)
    mean_scores = np.divide(scores.sum(axis=1), n_valid, out=np.zeros_like(n_valid, dtype=float), where=n_valid > 0)
    return float(np.max(mean_scores))

  s_ij = _directional(pae[np.ix_(mask_i, mask_j)])
  s_ji = _directional(pae[np.ix_(mask_j, mask_i)])
  return max(s_ij, s_ji)


def _calc_min_pae_pair(pae, mask_i, mask_j):
  """Symmetric inter-chain minPAE between two token masks."""
  if not np.any(mask_i) or not np.any(mask_j):
    return np.nan
  return float(min(np.min(pae[np.ix_(mask_i, mask_j)]), np.min(pae[np.ix_(mask_j, mask_i)])))


def _strip_struct_ext(path: pathlib.Path) -> str:
  name = path.name
  for ext in (".gz", ".cif", ".pdb"):
    name = name.removesuffix(ext)
  return name


def _parse_pae_metrics(model_path: pathlib.Path, mobile, pae_cutoff=10.0):
  """Computes ipSAE and minPAE for A–B and A–all glycan/ligand chains (non-A/B tokens)."""
  res = {"ipsae_AB": np.nan, "ipsae_AC": np.nan, "pae_min_AB": np.nan, "pae_min_AC": np.nan}
  parent, stem = model_path.parent, _strip_struct_ext(model_path)
  base_stem = stem.removesuffix("_model")

  pae, cids = None, None

  conf_json = next((
      p for p in (
          parent / f"{base_stem}_confidences.json",
          parent / "confidences.json",
      ) if p.exists()
  ), None)

  if conf_json is not None:
    data = json.loads(conf_json.read_text())
    if "pae" in data and "token_chain_ids" in data:
      pae = np.asarray(data["pae"], dtype=float)
      cids = np.asarray(data["token_chain_ids"], dtype=str)

  if pae is None:
    boltz_pae = parent / f"pae_{stem}.npz"
    if boltz_pae.exists():
      pae = np.asarray(np.load(boltz_pae)["pae"], dtype=float)
      is_prot = struc.filter_amino_acids(mobile)
      ca = mobile[is_prot & (mobile.atom_name == "CA")]
      lig = mobile[_ligand_mask(mobile)]
      cids_cand = np.concatenate([ca.chain_id.astype(str), lig.chain_id.astype(str)])
      if len(cids_cand) == pae.shape[0]:
        cids = cids_cand

  if pae is not None and cids is not None:
    mask_a = (cids == "A")
    mask_b = (cids == "B")
    mask_lig = ~np.isin(cids, ["A", "B"])
    res["ipsae_AB"] = _calc_ipsae_pair(pae, mask_a, mask_b, pae_cutoff)
    res["ipsae_AC"] = _calc_ipsae_pair(pae, mask_a, mask_lig, pae_cutoff)
    res["pae_min_AB"] = _calc_min_pae_pair(pae, mask_a, mask_b)
    res["pae_min_AC"] = _calc_min_pae_pair(pae, mask_a, mask_lig)
  return res


def _parse_confidence_json(model_path: pathlib.Path, has_lig: bool = True):
  parent, stem = model_path.parent, _strip_struct_ext(model_path)
  base_stem = stem.removesuffix("_model")
  conf_file = next((
      p for p in (
          parent / f"{base_stem}_summary_confidences.json",
          parent / "summary_confidences.json",
          parent / f"confidence_{stem}.json",
      ) if p.exists()
  ), None)

  res = {
      "iptm": np.nan, "ptm": np.nan, "binder_ptm": np.nan,
      "iptm_AB": np.nan, "iptm_AC": np.nan,
      "min_pae_AB": np.nan, "min_pae_AC": np.nan, "has_clash": 0,
  }
  if conf_file is None:
    return res

  data = json.loads(conf_file.read_text())
  if "chain_pair_iptm" in data:
    res["iptm"] = data.get("iptm", np.nan)
    res["ptm"] = data.get("ptm", np.nan)
    res["has_clash"] = int(data.get("has_clash", 0))
    c_ptm = data.get("chain_ptm", [])
    if len(c_ptm) >= 1 and c_ptm[0] is not None:
      res["binder_ptm"] = float(c_ptm[0])
    cp_iptm = data.get("chain_pair_iptm", [])
    if len(cp_iptm) >= 2 and cp_iptm[0][1] is not None:
      res["iptm_AB"] = float(cp_iptm[0][1])
    if has_lig and len(cp_iptm) >= 3:
      lig_iptms = [float(v) for v in cp_iptm[0][2:] if v is not None]
      if lig_iptms:
        res["iptm_AC"] = max(lig_iptms)
    cp_pae = data.get("chain_pair_pae_min", [])
    if len(cp_pae) >= 2 and cp_pae[0][1] is not None:
      res["min_pae_AB"] = float(cp_pae[0][1])
    if has_lig and len(cp_pae) >= 3:
      lig_paes = [float(v) for v in cp_pae[0][2:] if v is not None]
      if lig_paes:
        res["min_pae_AC"] = min(lig_paes)
  else:
    res["iptm"] = data.get("iptm", np.nan)
    res["ptm"] = data.get("ptm", np.nan)
    chains_ptm = data.get("chains_ptm", {})
    if "0" in chains_ptm:
      res["binder_ptm"] = float(chains_ptm["0"])
    pair_iptm = data.get("pair_chains_iptm", {})
    if "0" in pair_iptm:
      if "1" in pair_iptm["0"]:
        res["iptm_AB"] = float(pair_iptm["0"]["1"])
      if has_lig:
        lig_iptms = [float(v) for k, v in pair_iptm["0"].items() if k.isdigit() and int(k) >= 2 and v is not None]
        if lig_iptms:
          res["iptm_AC"] = max(lig_iptms)
    if has_lig and np.isnan(res["iptm_AC"]) and "ligand_iptm" in data:
      res["iptm_AC"] = float(data["ligand_iptm"])
    if np.isnan(res["iptm_AB"]) and "protein_iptm" in data:
      res["iptm_AB"] = float(data["protein_iptm"])
  return res


# Resolve pred_root (auto-fallback to pred_out from the refolding cell)
resolved_pred = pred_root.strip() or (str(pred_out) if "pred_out" in globals() else "")
if not resolved_pred:
  raise ValueError("No prediction directory specified. Run the refolding cell first or set pred_root.")
pred_path = pathlib.Path(resolved_pred).resolve()

# Collect metrics across all designs
rows = []
has_ligand = False
max_het_res = 0
for design_path in sorted(glob.glob(str(design_glob))):
  stem = _strip_struct_ext(pathlib.Path(design_path))

  candidates = (
      pred_path / stem, pred_path / stem.lower(),
      pred_path / f"{stem}.cif", pred_path / f"{stem}.cif".lower(),
      pred_path / "predictions" / stem, pred_path / "predictions" / f"{stem}.cif",
  )
  design_dir = next((d for d in candidates if d.is_dir()), None)
  if design_dir is not None:
    matches = [
        p for pat in ("*.cif", "*.cif.gz", "*.pdb")
        for p in sorted(design_dir.rglob(pat))
        if p.is_file() and "inputs" not in p.parts
    ]
  else:
    matches = [
        p for pat in (f"{stem}*model*.cif", f"{stem}*model*.cif.gz", f"{stem}*model*.pdb")
        for p in sorted(pred_path.rglob(pat))
        if p.is_file() and "inputs" not in p.parts
    ]

  if only_best:
    top_dir = design_dir if design_dir is not None else pred_path
    top_af3_rf3 = [m for m in matches if m.parent == top_dir and m.name.endswith(("_model.cif", "_model.cif.gz"))]
    top_boltz = [m for m in matches if m.name.endswith(("_model_0.cif", "_model_0.pdb"))]
    matches = (top_af3_rf3 or top_boltz or matches)[:1]
  else:
    samples = [m for m in matches if "sample-" in str(m) or "_model_" in m.name]
    matches = samples or matches

  if not matches:
    continue

  fixed = _load_struct(design_path)
  n_het = len(_get_het_residues(fixed))
  max_het_res = max(max_het_res, n_het)
  design_has_lig = n_het > 0
  has_ligand = has_ligand or design_has_lig
  cond_map = _load_cond_map(design_path)
  rfd3_hb = _count_hbonds(fixed, cond_map=cond_map)

  for m in matches:
    mobile = _load_struct(m)
    row = {"design": stem, "model": m.parent.name if m.name == "model.cif" else _strip_struct_ext(m)}
    row.update(_calc_rmsds(fixed, mobile))
    row["binder_plddt"] = _binder_plddt(mobile)
    row.update(_parse_confidence_json(m, has_lig=design_has_lig))

    pae_m = _parse_pae_metrics(m, mobile, pae_cutoff=ipsae_pae_cutoff)
    row["ipsae_AB"] = pae_m["ipsae_AB"]
    row["ipsae_AC"] = pae_m["ipsae_AC"] if design_has_lig else np.nan
    if np.isnan(row["min_pae_AB"]):
      row["min_pae_AB"] = pae_m["pae_min_AB"]
    if design_has_lig and np.isnan(row["min_pae_AC"]):
      row["min_pae_AC"] = pae_m["pae_min_AC"]

    row.update(_count_hbonds(mobile, cond_map=cond_map))
    row["rfd3_hbonds_AB"] = rfd3_hb["hbonds_AB"]
    row["rfd3_hbonds_AC"] = rfd3_hb["hbonds_AC"]
    row["rfd3_cond_hbonds"] = rfd3_hb["cond_hbonds"]
    rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
  raise FileNotFoundError(f"No matching design/prediction pairs found in {pred_path}. Check design_glob and pred_root.")

df["is_hit"] = (
    (df["binder_pose_rmsd"] <= pose_rmsd_cutoff) &
    (df["iptm_AB"].fillna(df["iptm"]) >= iptm_cutoff) &
    (df["binder_plddt"] >= plddt_cutoff)
)

sort_cols = ["is_hit", "ipsae_AB", "iptm_AC" if has_ligand else "iptm_AB", "binder_pose_rmsd"]
df = df.sort_values(by=sort_cols, ascending=[False, False, False, True]).reset_index(drop=True)

csv_path = pathlib.Path(f"{out_prefix}_metrics.csv").resolve()
csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(csv_path, index=False)

is_glycan = max_het_res > 1
het_noun = "Glycan" if is_glycan else "Ligand"
mode_label = f"Glycoprotein ({max_het_res} sugars)" if is_glycan else ("Protein + Ligand" if has_ligand else "Protein-Only")
print(f"[{mode_label}] Saved {len(df)} prediction(s) to {csv_path}  |  Hits passing cutoffs: {int(df['is_hit'].sum())}/{len(df)}")

if has_ligand:
  display_cols = [
      "design", "binder_pose_rmsd", "binder_fold_rmsd", "ligand_rmsd",
      "binder_plddt", "iptm_AB", "iptm_AC", "ipsae_AB", "ipsae_AC",
      "min_pae_AB", "min_pae_AC", "rfd3_hbonds_AB", "hbonds_AB", "hbonds_AC",
      "rfd3_cond_hbonds", "cond_hbonds", "is_hit",
  ]
else:
  display_cols = [
      "design", "binder_pose_rmsd", "binder_fold_rmsd", "target_rmsd",
      "binder_plddt", "iptm_AB", "ipsae_AB", "min_pae_AB",
      "rfd3_hbonds_AB", "hbonds_AB", "rfd3_cond_hbonds", "cond_hbonds", "is_hit",
  ]
display(df[display_cols].round(2))

# --- 6-Panel Publication-Quality Plot ---
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

fig, axes = plt.subplots(2, 3, figsize=(12.2, 7.2), dpi=300)
colors = np.where(df["is_hit"], "#54A24B", "#4C78A8")
iptm_ab = df["iptm_AB"].fillna(df["iptm"])
labels_legend = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#54A24B", markersize=7, label="Hit (passes cutoffs)"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#4C78A8", markersize=7, label="Other design"),
]

# Panel A: Binder Pose RMSD vs. Binder-Target ipTM (A-B)
ax = axes[0, 0]
ax.scatter(df["binder_pose_rmsd"], iptm_ab, c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
ax.axvline(pose_rmsd_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.axhline(iptm_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.set_xlabel("Binder Pose RMSD (Å)")
ax.set_ylabel("Binder–Target ipTM (A–B)")
ax.set_title("A. Docking vs. ipTM", fontsize=10, fontweight="bold", loc="left")

# Panel B: Binder Pose RMSD vs. Binder-Target ipSAE (A-B)
ax = axes[0, 1]
if df["ipsae_AB"].notna().any():
  ax.scatter(df["binder_pose_rmsd"], df["ipsae_AB"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
  ax.axvline(pose_rmsd_cutoff, color="#888888", linestyle="--", linewidth=0.9)
  ax.axhline(ipsae_cutoff, color="#888888", linestyle="--", linewidth=0.9)
else:
  ax.text(0.5, 0.5, "Full PAE matrix not found\n(needed for ipSAE)", ha="center", va="center", fontsize=9, color="#666666", transform=ax.transAxes)
ax.set_xlabel("Binder Pose RMSD (Å)")
ax.set_ylabel("Binder–Target ipSAE (A–B)")
ax.set_title("B. Docking vs. ipSAE", fontsize=10, fontweight="bold", loc="left")

# Panel C: Inter-chain minPAE
ax = axes[0, 2]
if has_ligand:
  if df["min_pae_AB"].notna().any() and df["min_pae_AC"].notna().any():
    ax.scatter(df["min_pae_AB"], df["min_pae_AC"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
    ax.axvline(min_pae_cutoff, color="#888888", linestyle="--", linewidth=0.9)
    ax.axhline(min_pae_cutoff, color="#888888", linestyle="--", linewidth=0.9)
  else:
    ax.text(0.5, 0.5, "minPAE not available\n(PAE matrix not found)", ha="center", va="center", fontsize=9, color="#666666", transform=ax.transAxes)
  ax.set_xlabel("Binder–Target minPAE (Å)")
  ax.set_ylabel(f"Binder–{het_noun} minPAE (Å)")
  ax.set_title("C. Inter-chain minPAE", fontsize=10, fontweight="bold", loc="left")
else:
  if df["min_pae_AB"].notna().any():
    ax.scatter(df["binder_pose_rmsd"], df["min_pae_AB"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
    ax.axvline(pose_rmsd_cutoff, color="#888888", linestyle="--", linewidth=0.9)
    ax.axhline(min_pae_cutoff, color="#888888", linestyle="--", linewidth=0.9)
  else:
    ax.text(0.5, 0.5, "minPAE not available\n(PAE matrix not found)", ha="center", va="center", fontsize=9, color="#666666", transform=ax.transAxes)
  ax.set_xlabel("Binder Pose RMSD (Å)")
  ax.set_ylabel("Binder–Target minPAE (Å)")
  ax.set_title("C. Docking vs. minPAE", fontsize=10, fontweight="bold", loc="left")

# Panel D: Monomer Foldability
ax = axes[1, 0]
ax.scatter(df["binder_fold_rmsd"], df["binder_plddt"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
ax.axvline(2.0, color="#888888", linestyle="--", linewidth=0.9)
ax.axhline(plddt_cutoff, color="#888888", linestyle="--", linewidth=0.9)
ax.set_xlabel("Binder Fold RMSD (Å)")
ax.set_ylabel("Binder Mean pLDDT")
ax.set_title("D. Monomer Foldability", fontsize=10, fontweight="bold", loc="left")

# Panel E: Protein-interface H-bond recovery (HBPLUS)
ax = axes[1, 1]
ax.scatter(df["rfd3_hbonds_AB"], df["hbonds_AB"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
max_hb = max(1, int(np.nanmax(np.concatenate([df["rfd3_hbonds_AB"].to_numpy(dtype=float), df["hbonds_AB"].to_numpy(dtype=float)]))))
ax.plot([0, max_hb], [0, max_hb], color="#888888", linestyle="--", linewidth=0.9)
ax.set_xlabel("RFD3 Interface H-bonds (A–B)")
ax.set_ylabel("Predicted Interface H-bonds (A–B)")
ax.set_title("E. Protein H-Bond Recovery (HBPLUS)", fontsize=10, fontweight="bold", loc="left")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

# Panel F: Glycan avoidance check (or ipTM vs H-bonds when no ligand)
ax = axes[1, 2]
if has_ligand:
  ax.scatter(df["hbonds_AB"], df["hbonds_AC"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
  ax.set_xlabel("Predicted Binder–Target H-bonds (A–B)")
  ax.set_ylabel(f"Predicted Binder–{het_noun} H-bonds")
  ax.set_title(f"F. {het_noun} Avoidance Check (HBPLUS)", fontsize=10, fontweight="bold", loc="left")
  ax.xaxis.set_major_locator(MaxNLocator(integer=True))
  ax.yaxis.set_major_locator(MaxNLocator(integer=True))
else:
  ax.scatter(df["hbonds_AB"], iptm_ab, c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
  ax.axhline(iptm_cutoff, color="#888888", linestyle="--", linewidth=0.9)
  ax.set_xlabel("Predicted Binder–Target H-bonds (A–B)")
  ax.set_ylabel("Binder–Target ipTM (A–B)")
  ax.set_title("F. HBPLUS H-Bonds vs. ipTM", fontsize=10, fontweight="bold", loc="left")
  ax.xaxis.set_major_locator(MaxNLocator(integer=True))

for ax in axes.flat:
  ax.tick_params(top=False, right=False)

fig.legend(handles=labels_legend, loc="upper center", ncol=2, frameon=False, fontsize=9, bbox_to_anchor=(0.5, 1.02))
plt.tight_layout(rect=[0, 0, 1, 0.97])

fig_path = pathlib.Path(f"{out_prefix}_summary.png").resolve()
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Figure saved to {fig_path}")

In [ ]:
#@title Extract & Plot RFD3 Backbone Metrics against AF3/OF3/Boltz-2/RF3 Metrics
rfd3_dir = "8S6Z_binder_design/rfd3_out" #@param {type:"string"}
eval_csv = "8S6Z_binder_design/binder_eval_metrics.csv" #@param {type:"string"}
out_prefix = "8S6Z_binder_design/rfd3_metrics" #@param {type:"string"}
#@markdown Comma-separated exact metric keys to plot (from RFD3 JSON or merged CSV). Leave blank to auto-select up to 6 (adapts to protein-only vs. protein+ligand).
plot_metrics = "rfd3_hbonds_AB, rfd3_cond_hbonds, hbonds_AB, n_clashing.ligand_min_distance, loop_fraction, radius_of_gyration" #@param {type:"string"}

import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

rfd3_path = pathlib.Path(rfd3_dir).resolve()
json_files = sorted(rfd3_path.glob("*.json"))
if not json_files:
  raise FileNotFoundError(f"No RFD3 .json metric files found in {rfd3_path}")

def _flatten_metrics(obj, prefix=""):
  """Flatten nested dict while preserving original metric field names."""
  flat = {}
  for k, v in obj.items():
    if isinstance(v, dict):
      sub_prefix = "" if k in ("metrics", "summary", "scores") else f"{prefix}{k}."
      flat.update(_flatten_metrics(v, sub_prefix))
    elif isinstance(v, (int, float)) and not isinstance(v, bool):
      flat[f"{prefix}{k}"] = float(v)
  return flat

rfd3_rows = []
for jf in json_files:
  data = json.loads(jf.read_text())
  row = {"design": jf.stem}
  row.update(_flatten_metrics(data))
  rfd3_rows.append(row)

df_rfd3 = pd.DataFrame(rfd3_rows)
rfd3_numeric = [c for c in df_rfd3.columns if c != "design" and df_rfd3[c].notna().any()]
print(f"Found {len(df_rfd3)} RFD3 JSON file(s) with {len(rfd3_numeric)} numeric metric(s):")
print(" ", ", ".join(rfd3_numeric))

# Merge with refolding evaluation CSV if available
eval_path = pathlib.Path(eval_csv).resolve()
if eval_path.exists():
  df_eval = pd.read_csv(eval_path)
  df = df_eval.merge(df_rfd3, on="design", how="left")
  has_refold = True
else:
  df = df_rfd3.copy()
  df["is_hit"] = False
  has_refold = False
  print(f"Note: {eval_path} not found — plotting RFD3 metrics alone.")

merged_csv = pathlib.Path(f"{out_prefix}_merged.csv").resolve()
df.to_csv(merged_csv, index=False)
print(f"Saved merged table to {merged_csv}")

has_ligand = (
    ("ligand_rmsd" in df.columns and df["ligand_rmsd"].notna().any())
    or ("hbonds_AC" in df.columns and df["hbonds_AC"].fillna(0).gt(0).any())
)

all_numeric = [c for c in df.columns if c not in ("design", "model", "is_hit") and pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().any()]

if plot_metrics.strip():
  requested = [m.strip() for m in plot_metrics.split(",") if m.strip()]
  missing = [m for m in requested if m not in all_numeric]
  if missing:
    print(f"Warning: these plot_metrics were not found: {missing}")
  chosen = [m for m in requested if m in all_numeric]
else:
  if has_ligand:
    priority_terms = (
        "n_clashing.ligand_min_distance",
        "n_clashing.ligand_clashes",
        "loop_fraction",
        "radius_of_gyration",
        "hbonds_AB",
        "hbonds_AC",
    )
  else:
    priority_terms = (
        "n_chainbreaks",
        "loop_fraction",
        "non_loop_fraction",
        "radius_of_gyration",
        "rfd3_hbonds_AB",
        "hbonds_AB",
    )
  chosen = [c for c in priority_terms if c in all_numeric][:6]
  for col in rfd3_numeric:
    if len(chosen) == 6:
      break
    if col not in chosen and df[col].nunique() > 1:
      chosen.append(col)

if not chosen:
  raise ValueError("No valid numeric metrics found to plot. Check the printed key list above.")

# --- Plot ---
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

n_panels = len(chosen)
ncols = 3 if n_panels >= 5 else (2 if n_panels > 1 else 1)
nrows = int(np.ceil(n_panels / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.1 * ncols, 3.5 * nrows), dpi=300, squeeze=False)
colors = np.where(df.get("is_hit", False), "#54A24B", "#4C78A8")

for ax, col in zip(axes.flat, chosen):
  if has_refold and "binder_pose_rmsd" in df.columns and df["binder_pose_rmsd"].notna().any():
    ax.scatter(df[col], df["binder_pose_rmsd"], c=colors, s=55, edgecolors="white", linewidths=0.6, alpha=0.9)
    ax.set_ylabel("Binder Pose RMSD (Å)")
  else:
    ax.hist(df[col].dropna(), bins=15, color="#4C78A8", edgecolor="white", alpha=0.9)
    ax.set_ylabel("Count")
  ax.set_xlabel(col)
  if any(k in col.lower() for k in ("hbond", "clash", "chainbreak", "num_")):
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
  ax.tick_params(top=False, right=False)

for ax in axes.flat[n_panels:]:
  ax.axis("off")

plt.tight_layout()
fig_path = pathlib.Path(f"{out_prefix}_plots.png").resolve()
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Figure saved to {fig_path}")